In [201]:
import pandas as pd
import numpy as np
import re
import ast
import json
from collections import Counter

In [202]:
import warnings
warnings.filterwarnings('ignore')

# Beko

In [203]:
path1 = "../web-scraping-data/data/raw/beko/beko_urun_ozellik.csv"
path2 = "../web-scraping-data/data/raw/beko/beko_urun_ozellik_2.csv"
path3 = "../web-scraping-data/data/raw/beko/beko_urun_ozellik3.csv"

df1 = pd.read_csv(path1)
df2 = pd.read_csv(path2)
df3 = pd.read_csv(path3)

print(f"df1 shape: {df1.shape}")
print(f"df2 shape: {df2.shape}")
print(f"df3 shape: {df3.shape}")


df1 shape: (1184, 10)
df2 shape: (126, 10)
df3 shape: (651, 10)


In [204]:
# beko_turleri.csv dosyasını kat_tur.csv olarak kaydet
beko_turleri_path = "../web-scraping-data/data/raw/beko/beko_turleri.csv"
kat_tur_df = pd.read_csv(beko_turleri_path)

print(kat_tur_df.head())

     kategori          urun_adi
0  Beyaz Eşya         Buzdolabı
1  Beyaz Eşya   Derin Dondurucu
2  Beyaz Eşya  Çamaşır Makinesi
3  Beyaz Eşya  Bulaşık Makinesi
4  Beyaz Eşya  Kurutma Makinesi


In [205]:
# birleştir ve indeksleri sıfırla, aynı ürünü tekilleştir
df = pd.concat([df1, df2, df3], ignore_index=True, sort=False)
df.drop_duplicates(subset='urun_linki', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"merged df shape: {df.shape}")
df.sample(5, random_state=15)

merged df shape: (1301, 10)


,urun_linki,urun_adi,fiyat,teknik_ozellikler,enerji_sinifi,favori_sayisi,yorum_sayisi,renk_secenekleri,urun_ozellikleri,yorumlar
94,https://www.beko.com.tr/12-kg-camasir-makinesi...,CMX 12140 12 Kg Çamaşır Makinesi,39.959 TL,"{""Kapasite"": ""12 kg"", ""Maksimum Sıkma Devri"": ...",A,120.00,NaN,[],"{""Kapasite"": ""12 kg"", ""Maksimum Sıkma Devri"": ...","[{""yildiz"": ""5"", ""baslik"": ""F/P ÜRÜNÜ"", ""sahip..."
482,https://www.beko.com.tr/kulaklik/jr310bt-bluet...,JR310BT Bluetooth ÇocukKulaklığı OEMavi Kulaklık,2.190 TL,"{""Kulaklık Tipi"": ""Kablosuz kulak üstü kulaklı...",NaN,0.00,NaN,[],"{""Kulaklık Tipi"": ""Kablosuz kulak üstü kulaklı...",[]
295,https://www.beko.com.tr/fitfry-buhar-destekli-...,BFC 450 A FitFry Buhar Destekli Fırın,59.405 TL,"{""Motor Teknolojisi"": ""AeroPerfect"", ""Homewhiz...",A+,5.00,NaN,[],"{""Motor Teknolojisi"": ""AeroPerfect"", ""Homewhiz...",[]
434,https://www.beko.com.tr/hoparlor/band-bt-hopar...,Grundig Band BT Hoparlör Yeşil Hoparlör,2.099 TL,"{""Ses Çıkış Gücü"": ""2x 2,5 W"", ""Çalma Süresi"":...",NaN,2.00,NaN,[],"{""Ses Çıkış Gücü"": ""2x 2,5 W"", ""Çalma Süresi"":...",[]
1015,https://www.beko.com.tr/kucuk-ev-aletleri-yede...,Bıçak Grubu Blender Küçük Ev Aletleri Yedek Pa...,475 TL,{},NaN,1.00,1,[],{},[]


In [206]:
df = df.drop_duplicates().reset_index(drop=True)
df = df.drop_duplicates(subset='urun_linki', keep='first').reset_index(drop=True)
print(f"df shape after removing duplicates: {df.shape}")

df shape after removing duplicates: (1301, 10)


In [207]:
df_yedek = df.copy()

In [208]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   urun_linki         1301 non-null   str    
 1   urun_adi           1301 non-null   str    
 2   fiyat              1301 non-null   str    
 3   teknik_ozellikler  1301 non-null   str    
 4   enerji_sinifi      270 non-null    object 
 5   favori_sayisi      1237 non-null   float64
 6   yorum_sayisi       126 non-null    object 
 7   renk_secenekleri   1301 non-null   str    
 8   urun_ozellikleri   1301 non-null   str    
 9   yorumlar           1301 non-null   str    
dtypes: float64(1), object(2), str(7)
memory usage: 1.2+ MB


In [209]:
# 1. Kategori Sütununu Çıkarma
# URL'i '/' işaretine göre böldüğümüzde liste şu şekilde olur: 
# ['https:', '', 'www.beko.com.tr', 'kategori_adi', 'urun_adi']
# Bize 3. indeksteki 'kategori_adi' lazım.
df['Product_Group'] = df['urun_linki'].str.split('/').str[3]

# 2. Tire (-) İşaretlerini Temizleme
# Tireleri boşluk karakteri ile değiştiriyoruz
df['Product_Group'] = df['Product_Group'].str.replace('-', ' ', regex=False)

# 3. Metin Formatını Düzeltme
# Her kelimenin ilk harfini büyük yapmak için (Örn: "oyun konsolu" -> "Oyun Konsolu")
df['Product_Group'] = df['Product_Group'].str.title()

# Sonucu kontrol etmek için
print(df[['urun_linki', 'Product_Group']].sample(5, random_state=15))

                                             urun_linki  \
94    https://www.beko.com.tr/12-kg-camasir-makinesi...   
482   https://www.beko.com.tr/kulaklik/jr310bt-bluet...   
295   https://www.beko.com.tr/fitfry-buhar-destekli-...   
434   https://www.beko.com.tr/hoparlor/band-bt-hopar...   
1015  https://www.beko.com.tr/kucuk-ev-aletleri-yede...   

                                      Product_Group  
94                           12 Kg Camasir Makinesi  
482                                        Kulaklik  
295                     Fitfry Buhar Destekli Firin  
434                                        Hoparlor  
1015  Kucuk Ev Aletleri Yedek Parca Ve Aksesuarlari  


In [210]:
unique_values = pd.Series(df["Product_Group"].dropna().unique()).sort_values()
print(unique_values.to_string(index=False))

                       10 Kg Camasir Makinesi
                       10 Kg Kurutma Makinesi
                       11 Kg Camasir Makinesi
                       11 Kg Kurutma Makinesi
                       12 Kg Camasir Makinesi
                       12 Kg Kurutma Makinesi
                    3 Bolmeli Derin Dondurucu
                 3 Programli Bulasik Makinesi
                    4 Bolmeli Derin Dondurucu
                 4 Programli Bulasik Makinesi
                    5 Bolmeli Derin Dondurucu
                 5 Programli Bulasik Makinesi
                    6 Bolmeli Derin Dondurucu
                 6 Programli Bulasik Makinesi
                    7 Bolmeli Derin Dondurucu
                   8 Bolmeli Derin Dondurucu 
                        8 Kg Camasir Makinesi
                        8 Kg Kurutma Makinesi
                        9 Kg Camasir Makinesi
                        9 Kg Kurutma Makinesi
                           Ada Tipi Davlumbaz
                                  

In [211]:

# Türkçe karakterleri İngilizce karşılıklarına çeviren ve küçük harf yapan fonksiyon
def metin_temizle(text):
    if pd.isna(text):
        return ""
    text = str(text).lower() 
    ceviri_tablosu = str.maketrans("çğıöşü", "cgiosu")
    text = text.translate(ceviri_tablosu)
    return " ".join(text.split())

# 1. Aşama: Senin Manuel Eşleştirmelerin ve Otomatik Eşleşmelerin Sözlüğü
# Sözlük Yapısı: {'Alt Kategori (Temizlenmiş)': ('Ana Kategori', 'Orijinal Alt Kategori Adı')}
kategori_kurallari = {
    # BEYAZ EŞYA GRUBU
    'camasir makinesi': ('Beyaz Eşya', 'Çamaşır Makinesi'),
    'kurutma makinesi': ('Beyaz Eşya', 'Kurutma Makinesi'),
    'kurutmali camasir makinesi': ('Beyaz Eşya', 'Kurutmalı Çamaşır Makinesi'),
    'bulasik makinesi': ('Beyaz Eşya', 'Bulaşık Makinesi'),
    'derin dondurucu': ('Beyaz Eşya', 'Derin Dondurucu'),
    'buzdolabi': ('Beyaz Eşya', 'Buzdolabı'),
    'firin': ('Beyaz Eşya', 'Fırın'),
    'su sebili': ('Beyaz Eşya', 'Su Sebili'),
    'beyaz esya yedek parca': ('Beyaz Eşya', 'Beyaz Eşya Yedek Parça ve Aksesuarları'),
    'mikrodalga': ('Beyaz Eşya', 'Mikrodalga Fırın'), # Senin Kuralın
    'set ustu ocak': ('Beyaz Eşya', 'Set Üstü Ocak'), # Senin Kuralın
    'domino ocak': ('Beyaz Eşya', 'Set Üstü Ocak'),
    'gazli ocak': ('Beyaz Eşya', 'Set Üstü Ocak'),
    'induksiyonlu ocak': ('Beyaz Eşya', 'Set Üstü Ocak'),
    'vitroseramik ocak': ('Beyaz Eşya', 'Set Üstü Ocak'),

    # ANKASTRE GRUBU
    'ankastre bulasik': ('Ankastre', 'Ankastre Bulaşık Makineleri'), # Senin Kuralın
    'ankastre buzdolabi': ('Ankastre', 'Ankastre Buzdolabı'),
    'ankastre firin': ('Ankastre', 'Ankastre Fırın'),
    'ankastre mikrodalga': ('Ankastre', 'Ankastre Mikrodalgalar'),
    'ankastre davlumbaz': ('Ankastre', 'Ankastre Davlumbazlar'),
    'ankastre aspirator': ('Ankastre', 'Ankastre Aspiratörler'),
    'ankastre set': ('Ankastre', 'Ankastre Set'), # Senin Kuralın
    'ankastre ocak': ('Ankastre', 'Ankastre Ocak'), # Senin Kuralın
    'pisirici yedek parca': ('Ankastre', 'Pişirici Yedek Parça ve Aksesuarları'),
    'ada tipi davlumbaz': ('Ankastre', 'Ankastre Davlumbazlar'),
    'duvar tipi davlumbaz': ('Ankastre', 'Ankastre Davlumbazlar'),
    'gomme aspirator': ('Ankastre', 'Ankastre Aspiratörler'),

    # ELEKTRONİK GRUBU
    'cep telefonu': ('Elektronik', 'Cep Telefonu'), # Senin Kuralın
    'iphone': ('Elektronik', 'Cep Telefonu'),
    'android': ('Elektronik', 'Cep Telefonu'),
    'giyilebilir': ('Elektronik', 'Giyilebilir Teknoloji'), # Senin Kuralın
    'akilli saat': ('Elektronik', 'Giyilebilir Teknoloji'),
    'akilli bileklik': ('Elektronik', 'Giyilebilir Teknoloji'),
    'bilgisayar': ('Elektronik', 'Bilgisayar'), # Senin Kuralın
    'laptop': ('Elektronik', 'Bilgisayar'),
    'tablet': ('Elektronik', 'Bilgisayar'),
    'ses goruntu': ('Elektronik', 'Ses& Görüntü Sistemleri'), # Senin Kuralın
    'hoparlor': ('Elektronik', 'Ses& Görüntü Sistemleri'),
    'kulaklik': ('Elektronik', 'Ses& Görüntü Sistemleri'),
    'oyun': ('Elektronik', 'Hobi - Oyun (Gaming) Ürünleri'), # Senin Kuralın
    'konsol': ('Elektronik', 'Hobi - Oyun (Gaming) Ürünleri'),
    'arac ici kamera': ('Elektronik', 'Araç İçi Kamera'), # Senin Kuralın
    'yazar kasa': ('Elektronik', 'Ödeme Sistemleri'), # Senin Kuralın
    'pos': ('Elektronik', 'Ödeme Sistemleri'),
    'cep telefonu aksesuarlari': ('Elektronik', 'Cep Telefonu Aksesuarları'),
    'powerbank': ('Elektronik', 'Cep Telefonu Aksesuarları'),
    'sarj cihazi': ('Elektronik', 'Cep Telefonu Aksesuarları'),
    
    # ISITMA SOĞUTMA VE ENERJİ GRUBU
    'klima': ('Isıtma Soğutma ve Enerji', 'Klima'),
    'vantilator': ('Isıtma Soğutma ve Enerji', 'Vantilatör'),
    'isitici': ('Elektronik', 'Elektrikli Isıtıcı'), # Senin Kuralın
    'radyator': ('Elektronik', 'Elektrikli Isıtıcı'), 
    'termosifon': ('Isıtma Soğutma ve Enerji', 'Termosifon'),
    'kombi': ('Isıtma Soğutma ve Enerji', 'Kombi'),
    'hava sogutucu': ('Isıtma Soğutma ve Enerji', 'Hava Soğutucu'),
    'nem alma': ('Isıtma Soğutma ve Enerji', 'Nem Alma Cihazı'),
    'isitma sogutma yedek parca': ('Isıtma Soğutma ve Enerji', 'Isıtma Soğutma Yedek Parça ve Aksesuarları'), # Senin Kuralın

    # KÜÇÜK EV ALETLERİ GRUBU
    'kucuk ev aletleri': ('Küçük Ev Aletleri', 'Küçük Ev Aletleri Seti'), # Senin Kuralın (Setleri)
    'supurge': ('Küçük Ev Aletleri', 'Elektrikli Süpürge'), # Senin Kuralın
    'frozen': ('Küçük Ev Aletleri', 'Frozen ve Slush Makinesi'), # Senin Kuralın
    'slush': ('Küçük Ev Aletleri', 'Frozen ve Slush Makinesi'),
    'kahve makinesi': ('Küçük Ev Aletleri', 'Kahve Makinesi'), # Senin Kuralın
    'espresso': ('Küçük Ev Aletleri', 'Kahve Makinesi'),
    'pisirici': ('Küçük Ev Aletleri', 'Pişirici'), # Senin Kuralın
    'airfryer': ('Küçük Ev Aletleri', 'Pişirici'),
    'fritoz': ('Küçük Ev Aletleri', 'Pişirici'),
    'tost makinesi': ('Küçük Ev Aletleri', 'Pişirici'),
    'ekmek kizartma': ('Küçük Ev Aletleri', 'Pişirici'),
    'karistirici': ('Küçük Ev Aletleri', 'Karıştırıcı& Doğrayıcı'), # Senin Kuralın
    'dograyici': ('Küçük Ev Aletleri', 'Karıştırıcı& Doğrayıcı'),
    'blender': ('Küçük Ev Aletleri', 'Karıştırıcı& Doğrayıcı'),
    'mikser': ('Küçük Ev Aletleri', 'Karıştırıcı& Doğrayıcı'),
    'kiyma makinesi': ('Küçük Ev Aletleri', 'Karıştırıcı& Doğrayıcı'),
    'hamur yogurma': ('Küçük Ev Aletleri', 'Karıştırıcı& Doğrayıcı'),
    'kisisel bakim': ('Küçük Ev Aletleri', 'Kişisel Bakım'), # Senin Kuralın
    'agiz bakim': ('Küçük Ev Aletleri', 'Kişisel Bakım'),
    'baskul': ('Küçük Ev Aletleri', 'Kişisel Bakım'),
    'sac duzlestirici': ('Küçük Ev Aletleri', 'Kişisel Bakım'),
    'sac masasi': ('Küçük Ev Aletleri', 'Kişisel Bakım'),
    'tiras makinesi': ('Küçük Ev Aletleri', 'Kişisel Bakım'),
    'outdoor': ('Küçük Ev Aletleri', 'Outdoor Ekipman'), # Senin Kuralın
    'termos': ('Küçük Ev Aletleri', 'Outdoor Ekipman'),
    'kahve ve sofra': ('Küçük Ev Aletleri', 'Kahve ve Sofra Ürünleri'), # Senin Kuralın
    'ev temizlik': ('Küçük Ev Aletleri', 'Ev Temizlik Ekipmanları ve Teknolojileri'), # Senin Kuralın
    'utu': ('Küçük Ev Aletleri', 'Ütü'),
    'cay makinesi': ('Küçük Ev Aletleri', 'Çay Makinesi'),
    'semaver': ('Küçük Ev Aletleri', 'Semaver'),
    'kettle': ('Küçük Ev Aletleri', 'Kettle'),
    'meyve sikacagi': ('Küçük Ev Aletleri', 'Katı Meyve Sıkacağı'),
    'narenciye': ('Küçük Ev Aletleri', 'Narenciye Sıkacağı'),
    'dondurma makinesi': ('Küçük Ev Aletleri', 'Dondurma Makinesi'),
    'buz makinesi': ('Küçük Ev Aletleri', 'Buz Makinesi'),
    'uv temizleme': ('Küçük Ev Aletleri', 'UV Temizleme Cihazı'),
    'yogurt': ('Küçük Ev Aletleri', 'Yoğurt &Kefir Makinesi'),
    'kefir': ('Küçük Ev Aletleri', 'Yoğurt &Kefir Makinesi'),

    # HİJYEN-AKSESUAR-OTO GRUBU
    'aktif hijyen': ('Hijyen-Aksesuar-Oto', 'Aktif Hijyen'), # Senin Kuralın
    'arac buzdolabi': ('Hijyen-Aksesuar-Oto', 'Araç Buzdolabı'),
    'aksesuar': ('Hijyen-Aksesuar-Oto', 'Aksesuarlar'),

    # SU ARITMA GRUBU
    'su aritma': ('Su Arıtma', 'Su Arıtma Cihazları') # Senin Kuralın
}

# 2. Aşama: Eşleştirme Fonksiyonu
def akilli_eslestir(Product_Group):
    # Ürün adını temizle (Örn: "10 Kg Camasir Makinesi" -> "10 kg camasir makinesi")
    temiz_ad = metin_temizle(Product_Group)
    
    # Kural sözlüğündeki anahtar kelimeleri (keys) tek tek dolaş
    for anahtar_kelime, (ana_kat, alt_kat) in kategori_kurallari.items():
        # Eğer sözlükteki anahtar kelime, temizlenmiş ürün adının içinde geçiyorsa
        # (Örn: "camasir makinesi" metni "10 kg camasir makinesi" içinde geçer)
        if re.search(r'\b' + re.escape(anahtar_kelime) + r'\b', temiz_ad):
            return pd.Series([ana_kat, alt_kat])
            
    # Eğer hiçbir kurala uymazsa
    return pd.Series(['Diğer', 'Diğer'])

# 3. Aşama: Fonksiyonu Uygulama ve Yeni Sütunları Oluşturma
# df'indeki 'kategori' sütununu baz alıyoruz (veya doğrudan 'Product_Group' sütununu da kullanabilirsin)
# Eğer senin scraping verinde 'Product_Group' daha açıklayıcıysa df['Product_Group'] kullanmak daha doğru sonuç verebilir.
df[['Main_Category', 'Subcategory']] = df['Product_Group'].apply(akilli_eslestir)

# Sonucu Kontrol Etme (Hangi ürünlerin eşleşemediğini görmek için)
eslesmeyenler = df[df['Main_Category'] == 'Diğer']
print(f"Eşleşmeyen ürün sayısı: {len(eslesmeyenler)}")
if len(eslesmeyenler) > 0:
    print("\nEşleşemeyen ilk 10 ürün:")
    print(eslesmeyenler[['Product_Group']].head(10))

# Eşleşenlerden bir örnek
print("\nEşleşen ürünler örneği:")
print(df[['Product_Group', 'Main_Category', 'Subcategory']].sample(10, random_state=15))

Eşleşmeyen ürün sayısı: 174

Eşleşemeyen ilk 10 ürün:
                         Product_Group
168             Ankastre Mikrodalgalar
188             Gazli Cam Tablali Ocak
189             Gazli Cam Tablali Ocak
190        Entegre Davlumbazli Ocaklar
191        Entegre Davlumbazli Ocaklar
192               Vitroseramik Ocaklar
193  Gazli Inoks Metal Tablali Ocaklar
194             Gazli Cam Tablali Ocak
195                     Domino Ocaklar
196  Gazli Inoks Metal Tablali Ocaklar

Eşleşen ürünler örneği:
                                      Product_Group             Main_Category  \
94                           12 Kg Camasir Makinesi                Beyaz Eşya   
482                                        Kulaklik                Elektronik   
295                     Fitfry Buhar Destekli Firin                Beyaz Eşya   
434                                        Hoparlor                Elektronik   
1015  Kucuk Ev Aletleri Yedek Parca Ve Aksesuarlari         Küçük Ev Aletleri   
262   

In [212]:
df.sample(5, random_state=42)

,urun_linki,urun_adi,fiyat,teknik_ozellikler,enerji_sinifi,favori_sayisi,yorum_sayisi,renk_secenekleri,urun_ozellikleri,yorumlar,Product_Group,Main_Category,Subcategory
478,https://www.beko.com.tr/kulaklik/samsung-typec...,Samsung TypeC Kablolu Kulaklık-Siyah Kulaklık,599 TL,"{""Kulaklık Tipi"": ""Kablolu Kulak İçi Kulaklık""...",NaN,1.00,NaN,[],"{""Kulaklık Tipi"": ""Kablolu Kulak İçi Kulaklık""...",[],Kulaklik,Elektronik,Ses& Görüntü Sistemleri
722,https://www.beko.com.tr/cep-telefonu-aksesuarl...,UGREEN USB-C 60W Örgülü Kablo Syh 1M Cep Telef...,379 TL,"{""2 cm"": ""Derinlik"", ""4 cm"": ""Genişlik"", ""5 cm...",NaN,0.00,NaN,[],{},[],Cep Telefonu Aksesuarlari,Elektronik,Cep Telefonu
312,https://www.beko.com.tr/ankastre-mikrodalgalar...,BMM 2020 I Ankastre Mikrodalgalar,16.420 TL,"{""Mikrodalga Hacmi"": ""20"", ""Ürün Rengi"": ""İnok...",NaN,22.00,NaN,[],"{""Mikrodalga Hacmi"": ""20"", ""Ürün Rengi"": ""İnok...",[],Ankastre Mikrodalgalar,Diğer,Diğer
661,https://www.beko.com.tr/android-telefon-modell...,Xiaomi Redmi Note 14 Pro 12/512GB Siyah Androi...,20.999 TL,"{""Dahili Hafıza"": ""512GB"", ""Ön Kamera"": ""32MP""...",NaN,1.00,NaN,[],"{""Dahili Hafıza"": ""512GB"", ""Ön Kamera"": ""32MP""...",[],Android Telefon Modelleri,Elektronik,Cep Telefonu
969,https://www.beko.com.tr/espresso-makinesi/nesp...,Nespresso Aeroccino 4 Süt Köpürtücü Espresso M...,5.999 TL,"{""10 cm"": ""Genişlik"", ""17 cm"": ""Yükseklik"", ""A...",NaN,0.00,NaN,[],{},[],Espresso Makinesi,Küçük Ev Aletleri,Kahve Makinesi


In [213]:
# Tüm sütunları gez ve boş liste/sözlük yapılarını NaN yap
for col in df.columns:
    df[col] = df[col].apply(lambda x: np.nan if str(x).strip() in ['[]', '{}', ''] else x)

# Fiyat
# 1. 'TL' yazısını ve etrafındaki her türlü boşluğu (\n dahil) regex ile temizle
df['fiyat'] = df['fiyat'].str.replace(r'\s*TL', '', regex=True)

# 2. Binlik ayracı olan noktayı (.) tamamen sil (örn: 39.959 -> 39959)
df['fiyat'] = df['fiyat'].str.replace('.', '', regex=False)

# 3. Eğer başka satırlarda kuruş varsa (örn: 475,50), virgülü ondalık formata (.) çevir
df['fiyat'] = df['fiyat'].str.replace(',', '.', regex=False)

# 4. Temizlenmiş metni artık güvenle float tipine dönüştür
df['fiyat'] = pd.to_numeric(df['fiyat'], errors='coerce')


# Bilimsel gösterimi kapat, sayıları virgülden sonra 2 basamaklı normal formatta göster
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Favori Sayısı
# Metinleri temizle ve sayıya çevir
df['favori_sayisi'] = pd.to_numeric(df['favori_sayisi'], errors='coerce')

# NaN'ları bozmadan tam sayıya çevir (Büyük 'I' harfine dikkat!)
df['favori_sayisi'] = df['favori_sayisi'].astype('Int64')

# NumPy için bilimsel gösterimi kapat (suppress=True)
np.set_printoptions(suppress=True)

# Çıktıyı tekrar kontrol et
print(df['fiyat'].unique())
print(df['favori_sayisi'].unique())

[ 88409.         nan  15159.    57969.    22189.    34829.    19749.
  59129.    28279.    18899.    42409.    48579.    22989.    47839.
  34129.    31389.    39449.    35829.    50489.    36029.    33839.
  36809.    37769.    33869.    40619.    39809.    29489.    38639.
  40259.    32189.    35279.    46799.    39959.    37649.    41069.
  38489.    24479.    25979.    28079.    23729.    34859.    24029.
  33569.    29699.    21179.    23579.    26549.    36149.    30989.
  36869.    23369.    21089.    27899.    27209.    21869.    38999.
  39479.    40379.    39719.    45329.    40799.    38249.    35699.
  23999.    44129.    41849.    43229.    43499.    39749.    37139.
  40499.     8739.    28276.    46466.    15363.    12599.    28734.
  40039.    27982.    45093.    15120.    14700.     9540.    12600.
   5010.     9000.    10260.     5899.     7469.     6599.    11217.
   8119.    15577.    31701.    21499.    12999.     7749.     9949.
  12399.    16849.      525.      

In [214]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   urun_linki         1301 non-null   str    
 1   urun_adi           1301 non-null   str    
 2   fiyat              1176 non-null   float64
 3   teknik_ozellikler  1129 non-null   str    
 4   enerji_sinifi      270 non-null    str    
 5   favori_sayisi      1237 non-null   Int64  
 6   yorum_sayisi       126 non-null    str    
 7   renk_secenekleri   80 non-null     str    
 8   urun_ozellikleri   901 non-null    str    
 9   yorumlar           175 non-null    str    
 10  Product_Group      1301 non-null   str    
 11  Main_Category      1301 non-null   str    
 12  Subcategory        1301 non-null   str    
dtypes: Int64(1), float64(1), str(11)
memory usage: 1.3 MB


In [215]:
df = df.drop(columns=['yorumlar', 'renk_secenekleri', 'yorum_sayisi', 'urun_ozellikleri'])

# favori sayısı nan yap
df['favori_sayisi'] = df['favori_sayisi'].fillna(pd.NA).astype('Int64')

In [216]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   urun_linki         1301 non-null   str    
 1   urun_adi           1301 non-null   str    
 2   fiyat              1176 non-null   float64
 3   teknik_ozellikler  1129 non-null   str    
 4   enerji_sinifi      270 non-null    str    
 5   favori_sayisi      1237 non-null   Int64  
 6   Product_Group      1301 non-null   str    
 7   Main_Category      1301 non-null   str    
 8   Subcategory        1301 non-null   str    
dtypes: Int64(1), float64(1), str(7)
memory usage: 975.1 KB


In [217]:
df["teknik_ozellikler"].sample(5, random_state=15)

94      {"Kapasite": "12 kg", "Maksimum Sıkma Devri": ...
482     {"Kulaklık Tipi": "Kablosuz kulak üstü kulaklı...
295     {"Motor Teknolojisi": "AeroPerfect", "Homewhiz...
434     {"Ses Çıkış Gücü": "2x 2,5 W", "Çalma Süresi":...
1015                                                  NaN
Name: teknik_ozellikler, dtype: str

In [218]:
import pandas as pd
import ast

def parse_dict(val):
    if pd.isna(val):
        return {}
    try:
        parsed_val = ast.literal_eval(val)
        # Sadece gerçekten sözlük olanları alalım
        if isinstance(parsed_val, dict):
            return parsed_val
        return {}
    except (ValueError, SyntaxError):
        return {}

# Sözlük hallerini içeren geçici sütunu oluşturalım
df['teknik_ozellikler_dict'] = df['teknik_ozellikler'].apply(parse_dict)

In [219]:
from collections import Counter

# Her bir sözlüğün içindeki tüm anahtarları (key) tek bir listede topluyoruz
tum_anahtarlar = [key for d in df['teknik_ozellikler_dict'] for key in d.keys()]

# Counter ile bu listedeki elemanların kaçar kez geçtiğini saydırıyoruz
frekanslar = Counter(tum_anahtarlar)

print(f"Toplam {len(frekanslar)} farklı özellik bulundu.\n")
print("Özellik Adı - Bulunma Sayısı")
print("-" * 30)

# En çok tekrar edenden en aza doğru yazdırıyoruz
for özellik, adet in frekanslar.most_common():
    print(f"{özellik}: {adet}")

Toplam 1083 farklı özellik bulundu.

Özellik Adı - Bulunma Sayısı
------------------------------
Ağırlık: Paketsiz: 592
Boyut (cm) (GxYxD): 566
Derinlik: 565
Ürün Rengi: 496
Bluetooth: 279
Enerji Sınıfı: 245
Ekran Tipi: 227
Renk: 224
Ekran Boyutu: 215
İşletim Sistemi: 207
1 cm: 189
Ön Kamera: 188
Wi-Fi: 185
Dahili Hafıza: 175
Marka: 175
2.Arka Kamera: 174
60 cm: 165
8 cm: 164
NFC: 162
Ürün Tipi: 159
İşlemci: 151
Güç: 149
Ses Seviyesi: 143
İşlemci Çekirdek Sayısı: 141
4 cm: 139
Ekran Çözünürlüğü: 137
Kamera Zoom: 135
Motor Tipi: 131
7 cm: 131
Ön Kamera Flaş: 131
15 cm: 126
Görüntülü Konuşma: 126
Arka Kamera: 116
GPS: 116
RAM Kapasitesi: 108
5 cm: 107
Kontrol Sistemi: 100
Mikrofon: 99
Genişlik: 98
Yıllık Enerji Tüketimi (kWh): 93
Ses Kayıt: 90
Kapasite: 87
Yüz Haritalama: 87
16 cm: 81
Kulaklık Tipi: 80
İklim Sınıfı: 78
Toplam Hacim (L): 78
Ses Seviyesi Sınıfı: 78
Soğutma Sistemi: 78
Günlük Enerji Tüketimi (kwh/Gün): 78
Günlük enerji tüketimi 16°C (kWh/24h) (EU_2021_EP): 78
Dondurucu Yeri

In [220]:
# 1. frekanslar sözlüğündeki anahtarlardan isminde sayı (rakam) geçenleri filtreliyoruz
sayi_iceren_anahtarlar = [key for key in frekanslar.keys() if any(c.isdigit() for c in str(key))]

print(f"İsminde sayı geçen toplam {len(sayi_iceren_anahtarlar)} özellik bulundu.\n")
print("=" * 40)

# 2. Sadece bu şüpheli anahtarların aldıkları değerleri ve frekanslarını yazdırıyoruz
for anahtar in sayi_iceren_anahtarlar:
    # df içindeki sözlükleri dönüp, sadece bu anahtara sahip olanların değerlerini çekiyoruz
    degerler = [d[anahtar] for d in df['teknik_ozellikler_dict'] if anahtar in d]
    
    # Değerlerin frekansını çıkarıyoruz (Counter'ı zaten import etmiştik)
    deger_frekanslari = Counter(degerler)
    
    # Çıktıyı yazdırıyoruz
    print(f"ÖZELLİK ADI: '{anahtar}' (Toplam {len(degerler)} üründe var)")
    for deger, adet in deger_frekanslari.most_common():
        print(f"   -> Aldığı Değer: {deger} (Frekans: {adet})")
    print("-" * 40)

İsminde sayı geçen toplam 186 özellik bulundu.

ÖZELLİK ADI: '76 cm' (Toplam 23 üründe var)
   -> Aldığı Değer: Derinlik (Frekans: 22)
   -> Aldığı Değer: Genişlik (Frekans: 1)
----------------------------------------
ÖZELLİK ADI: '83 cm' (Toplam 10 üründe var)
   -> Aldığı Değer: Genişlik (Frekans: 10)
----------------------------------------
ÖZELLİK ADI: '193 cm' (Toplam 4 üründe var)
   -> Aldığı Değer: Yükseklik (Frekans: 4)
----------------------------------------
ÖZELLİK ADI: 'Günlük enerji tüketimi 16°C (kWh/24h) (EU_2021_EP)' (Toplam 78 üründe var)
   -> Aldığı Değer: 0.48 (Frekans: 9)
   -> Aldığı Değer: 0.438 (Frekans: 6)
   -> Aldığı Değer: 0.384 (Frekans: 4)
   -> Aldığı Değer: 0.38 (Frekans: 4)
   -> Aldığı Değer: 0.4 (Frekans: 3)
   -> Aldığı Değer: 0.49 (Frekans: 3)
   -> Aldığı Değer: 0.447 (Frekans: 3)
   -> Aldığı Değer: 0.579 (Frekans: 3)
   -> Aldığı Değer: 0.51 (Frekans: 3)
   -> Aldığı Değer: 0.403 (Frekans: 3)
   -> Aldığı Değer: 0.349 (Frekans: 3)
   -> Aldığı D

In [221]:
def sozluk_duzelt(eski_sozluk):
    yeni_sozluk = {}
    
    # Ters yazıldığından emin olduğumuz değerleri bir listeye alalım
    ters_degerler = ['Derinlik', 'Genişlik', 'Yükseklik']
    
    for key, value in eski_sozluk.items():
        # Veri tiplerini string'e çevirip sağdaki/soldaki boşlukları temizleyelim (garanti olsun)
        temiz_value = str(value).strip()
        temiz_key = str(key).strip()
        
        # Eğer değer 'Derinlik', 'Genişlik' veya 'Yükseklik' ise yer değiştir
        if temiz_value in ters_degerler:
            yeni_sozluk[temiz_value] = temiz_key
        else:
            # Değilse (Program, Enerji vb.) hiçbir şeye dokunmadan yeni sözlüğe ekle
            yeni_sozluk[key] = value
            
    return yeni_sozluk

# Yeni ve temizlenmiş sözlüklerimizi aynı sütunun üzerine yazalım 
# (İstersen kaybetmemek için 'teknik_ozellikler_temiz' diye yeni bir sütun da açabilirsin)
df['teknik_ozellikler_dict'] = df['teknik_ozellikler_dict'].apply(sozluk_duzelt)

In [222]:
from collections import Counter

# 1. GÜNCEL sözlüklerden tüm anahtarları yeniden topluyoruz
guncel_anahtarlar = [key for d in df['teknik_ozellikler_dict'] for key in d.keys()]
guncel_frekanslar = Counter(guncel_anahtarlar)

# 2. İsminde sayı (rakam) geçen anahtarları YENİDEN filtreliyoruz
sayi_iceren_anahtarlar_kalan = [key for key in guncel_frekanslar.keys() if any(c.isdigit() for c in str(key))]

print(f"Temizlik sonrası isminde sayı geçen toplam {len(sayi_iceren_anahtarlar_kalan)} özellik kaldı.\n")
print("=" * 40)

# 3. Kalan anahtarların aldıkları değerleri ve frekanslarını yazdırıyoruz
for anahtar in sayi_iceren_anahtarlar_kalan:
    # df içindeki GÜNCEL sözlükleri dönüp, değerleri çekiyoruz
    degerler = [d[anahtar] for d in df['teknik_ozellikler_dict'] if anahtar in d]
    
    deger_frekanslari = Counter(degerler)
    
    print(f"ÖZELLİK ADI: '{anahtar}' (Toplam {len(degerler)} üründe var)")
    for deger, adet in deger_frekanslari.most_common():
        print(f"   -> Aldığı Değer: {deger} (Frekans: {adet})")
    print("-" * 40)

Temizlik sonrası isminde sayı geçen toplam 49 özellik kaldı.

ÖZELLİK ADI: 'Günlük enerji tüketimi 16°C (kWh/24h) (EU_2021_EP)' (Toplam 78 üründe var)
   -> Aldığı Değer: 0.48 (Frekans: 9)
   -> Aldığı Değer: 0.438 (Frekans: 6)
   -> Aldığı Değer: 0.384 (Frekans: 4)
   -> Aldığı Değer: 0.38 (Frekans: 4)
   -> Aldığı Değer: 0.4 (Frekans: 3)
   -> Aldığı Değer: 0.49 (Frekans: 3)
   -> Aldığı Değer: 0.447 (Frekans: 3)
   -> Aldığı Değer: 0.579 (Frekans: 3)
   -> Aldığı Değer: 0.51 (Frekans: 3)
   -> Aldığı Değer: 0.403 (Frekans: 3)
   -> Aldığı Değer: 0.349 (Frekans: 3)
   -> Aldığı Değer: 0.449 (Frekans: 2)
   -> Aldığı Değer: 0.46 (Frekans: 2)
   -> Aldığı Değer: 0.463 (Frekans: 2)
   -> Aldığı Değer: 0.494 (Frekans: 2)
   -> Aldığı Değer: 0.34 (Frekans: 2)
   -> Aldığı Değer: 0.382 (Frekans: 2)
   -> Aldığı Değer: 0.331 (Frekans: 1)
   -> Aldığı Değer: 0.451 (Frekans: 1)
   -> Aldığı Değer: 0.18 (Frekans: 1)
   -> Aldığı Değer: 0.178 (Frekans: 1)
   -> Aldığı Değer: 0.146 (Frekans: 1)


In [223]:
def program_fonksiyon_liste_yap(eski_sozluk):
    yeni_sozluk = {}
    programlar = []
    fonksiyonlar = []
    
    for key, value in eski_sozluk.items():
        # Anahtarı küçük harfe çevirip boşluklarını silelim
        temiz_key = str(key).strip().lower()
        
        # Eğer anahtarın İÇİNDE 'program' geçiyorsa listeye ekle
        if 'program' in temiz_key:
            programlar.append(str(value).strip())
            
        # Eğer anahtarın İÇİNDE 'fonksiyon' geçiyorsa listeye ekle
        elif 'fonksiyon' in temiz_key:
            fonksiyonlar.append(str(value).strip())
            
        # İkisi de değilse aynen bırak
        else:
            yeni_sozluk[key] = value
            
    # Eğer programlar listesi boş değilse, doğrudan liste formatında sözlüğe ekle
    if programlar:
        yeni_sozluk['Programlar'] = programlar
        
    # Eğer fonksiyonlar listesi boş değilse, doğrudan liste formatında sözlüğe ekle
    if fonksiyonlar:
        yeni_sozluk['Fonksiyonlar'] = fonksiyonlar
        
    return yeni_sozluk

# Fonksiyonu DataFrame'e uygulayalım
df['teknik_ozellikler_dict'] = df['teknik_ozellikler_dict'].apply(program_fonksiyon_liste_yap)

In [224]:
from collections import Counter

# --- BÖLÜM 1: İÇİNDE SAYI KALAN ANAHTARLARIN KONTROLÜ ---
guncel_anahtarlar = [key for d in df['teknik_ozellikler_dict'] for key in d.keys()]
guncel_frekanslar = Counter(guncel_anahtarlar)

sayi_iceren_anahtarlar_kalan = [key for key in guncel_frekanslar.keys() if any(c.isdigit() for c in str(key))]

print(f"Temizlik sonrası isminde sayı geçen toplam {len(sayi_iceren_anahtarlar_kalan)} özellik kaldı.\n")
print("=" * 40)

for anahtar in sayi_iceren_anahtarlar_kalan:
    degerler = []
    for d in df['teknik_ozellikler_dict']:
        if anahtar in d:
            val = d[anahtar]
            # Eğer değer bir liste ise Counter hata vermesin diye tuple'a çeviriyoruz (garanti olsun)
            if isinstance(val, list):
                val = tuple(val)
            degerler.append(val)
    
    deger_frekanslari = Counter(degerler)
    
    print(f"ÖZELLİK ADI: '{anahtar}' (Toplam {len(degerler)} üründe var)")
    # Çıktı çok uzamasın diye en sık geçen ilk 5 değeri gösteriyoruz, istersen .most_common() içini boş bırakabilirsin
    for deger, adet in deger_frekanslari.most_common(5):
        print(f"   -> Aldığı Değer: {deger} (Frekans: {adet})")
    print("-" * 40)


# --- BÖLÜM 2: YENİ 'Programlar' VE 'Fonksiyonlar' LİSTELERİNİN KONTROLÜ ---
print("\n\n" + "=" * 50)
print("YENİ OLUŞTURULAN 'Programlar' VE 'Fonksiyonlar' KONTROLÜ")
print("=" * 50)

ornek_sayisi = 0
for i, d in enumerate(df['teknik_ozellikler_dict']):
    # İçinde Programlar veya Fonksiyonlar anahtarı olan ilk 5 ürünü yazdıralım
    if 'Programlar' in d or 'Fonksiyonlar' in d:
        print(f"Satır Index: {df.index[i]}")
        if 'Programlar' in d:
            print(f"  Programlar: {d['Programlar']}")
        if 'Fonksiyonlar' in d:
            print(f"  Fonksiyonlar: {d['Fonksiyonlar']}")
        print("-" * 30)
        
        ornek_sayisi += 1
        if ornek_sayisi == 5:
            break

Temizlik sonrası isminde sayı geçen toplam 27 özellik kaldı.

ÖZELLİK ADI: 'Günlük enerji tüketimi 16°C (kWh/24h) (EU_2021_EP)' (Toplam 78 üründe var)
   -> Aldığı Değer: 0.48 (Frekans: 9)
   -> Aldığı Değer: 0.438 (Frekans: 6)
   -> Aldığı Değer: 0.384 (Frekans: 4)
   -> Aldığı Değer: 0.38 (Frekans: 4)
   -> Aldığı Değer: 0.4 (Frekans: 3)
----------------------------------------
ÖZELLİK ADI: 'Su Tüketimi (1 Çevrim)' (Toplam 19 üründe var)
   -> Aldığı Değer: 49 L (Frekans: 5)
   -> Aldığı Değer: 51 L (Frekans: 3)
   -> Aldığı Değer: 52 L (Frekans: 3)
   -> Aldığı Değer: 44 L (Frekans: 2)
   -> Aldığı Değer: 56 L (Frekans: 2)
----------------------------------------
ÖZELLİK ADI: 'Enerji Tüketimi (100 Çevrim)' (Toplam 19 üründe var)
   -> Aldığı Değer: 49 kWh (Frekans: 5)
   -> Aldığı Değer: 51 kWh (Frekans: 3)
   -> Aldığı Değer: 46 kWh (Frekans: 3)
   -> Aldığı Değer: 44 kWh (Frekans: 2)
   -> Aldığı Değer: 54 kWh (Frekans: 2)
----------------------------------------
ÖZELLİK ADI: 'Çık

In [225]:
from collections import Counter
import pandas as pd

# 1. GÜNCEL FREKANSLARI HESAPLA VE TUTULACAKLARI BELİRLE
tum_anahtarlar = [key for d in df['teknik_ozellikler_dict'] for key in d.keys()]
frekanslar = Counter(tum_anahtarlar)

# Sadece frekansı 5'ten BÜYÜK olan anahtarları bir kümeye (set) alıyoruz (hızlı arama için)
tutulacak_anahtarlar = {key for key, adet in frekanslar.items() if adet > 5}

silinen_sayisi = len(frekanslar) - len(tutulacak_anahtarlar)
print(f"Toplam {len(frekanslar)} benzersiz özellik vardı.")
print(f"Frekansı 5 ve altında olan {silinen_sayisi} özellik siliniyor...")
print(f"Geriye {len(tutulacak_anahtarlar)} adet sağlam özellik (sütun adayı) kaldı.\n")


# 2. SÖZLÜKLERİ FİLTRELE (Nadir olanları uçur)
def nadir_ozellikleri_sil(eski_sozluk):
    # Sadece 'tutulacak_anahtarlar' içinde olan key'leri yeni sözlüğe aktar
    return {k: v for k, v in eski_sozluk.items() if k in tutulacak_anahtarlar}

df['teknik_ozellikler_dict'] = df['teknik_ozellikler_dict'].apply(nadir_ozellikleri_sil)


# 3. SÖZLÜKLERİ SÜTUNLARA PARÇALA VE ANA DF İLE BİRLEŞTİR
# pd.json_normalize sözlükleri alır ve hızlıca DataFrame sütunlarına çevirir
yeni_sutunlar_df = pd.json_normalize(df['teknik_ozellikler_dict'])

# Satırların kaymaması için orijinal DataFrame'in indexini yeni oluşan DataFrame'e kopyalıyoruz
yeni_sutunlar_df.index = df.index

# Orijinal df ile yeni oluşan sütunları yan yana birleştiriyoruz (axis=1)
df = pd.concat([df, yeni_sutunlar_df], axis=1)

print("İşlem tamam! Sözlükler başarıyla sütunlara ayrıldı.")
print("Eklenen yeni sütunlardan bazıları:")
print(list(yeni_sutunlar_df.columns)[:10]) # İlk 10 sütun adını gösterelim

Toplam 865 benzersiz özellik vardı.
Frekansı 5 ve altında olan 349 özellik siliniyor...
Geriye 516 adet sağlam özellik (sütun adayı) kaldı.

İşlem tamam! Sözlükler başarıyla sütunlara ayrıldı.
Eklenen yeni sütunlardan bazıları:
['Dondurucu Bölme Teknolojisi', 'Derinlik', 'Genişlik', 'Yükseklik', 'Sıcaklık Yükselme Süresi (Saat)', 'Ürün Rengi', 'Dondurucu Yeri', 'Elektronik Gösterge', 'Kontrol Sistemi', 'Tatil Modu']


In [226]:
df.drop(columns=['teknik_ozellikler', 'teknik_ozellikler_dict'], inplace=True)

In [227]:
# 1. Eksik değerlerin yüzdelik oranını hesapla
eksik_yuzde = df.isnull().mean() * 100

# 2. Hata vermeden (listeleri de destekleyerek) eşsiz değerleri sayan ufak bir fonksiyon
def essiz_sayiyi_bul(sutun):
    try:
        # Önce standart yöntemi denesin
        return sutun.nunique()
    except TypeError:
        # Eğer sütunda liste varsa (TypeError verirse), listeleri tuple'a çevirip öyle saysın
        return sutun.apply(lambda x: tuple(x) if isinstance(x, list) else x).nunique()

# Bu fonksiyonu tüm dataframe'e uygula
essiz_sayi = df.apply(essiz_sayiyi_bul)

# 3. İkisini tek bir DataFrame'de birleştirerek tabloyu oluştur
ozet_tablo = pd.DataFrame({
    'Eksik Değer Oranı (%)': eksik_yuzde.round(2),
    'Eşsiz Değer Sayısı': essiz_sayi
})

# En çok boşluk olan (sorunlu) sütunları en üstte görmek için sırala
ozet_tablo = ozet_tablo.sort_values(by='Eksik Değer Oranı (%)', ascending=False)

# Limitleri kaldır ve göster
pd.set_option('display.max_rows', None) 
ozet_tablo

,Eksik Değer Oranı (%),Eşsiz Değer Sayısı
Hayvan Tüyü Fırçası,99.54,1
Ütü Tipi,99.54,1
Susuz Çalışma Emniyeti,99.54,1
Doluluk göstergesi,99.54,3
Titanyum Emaye Kaplamalı Kazan,99.54,1
Yüksek Basınç Emniyet Valfi,99.54,1
IP Sınıfı,99.54,1
Enerji sınıfı (WH) (EP),99.54,1
İzolasyon,99.54,1
Sağ Ön Yanıcı Gücü,99.54,1


In [228]:
df.shape

(1301, 524)

In [229]:
# Sütun adını senin veri setindeki orijinal adına (Subcategory) güncelliyoruz
kategori_sutunu = 'Subcategory'

# 1. Her bir alt kategori için sütunların boşluk oranını hesapla 
# (apply yerine doğrudan isnull() üzerinden groupby yapmak çok daha hızlıdır)
kategori_bazli_bosluk = df.isnull().groupby(df[kategori_sutunu]).mean() * 100

# 2. Her bir sütun için "en düşük" boşluk oranını bul (Yani o sütunun en dolu olduğu kategoriyi baz al)
minimum_bosluk_oranlari = kategori_bazli_bosluk.min()

# 3. Kendi belirlediğin sınır: En dolu olduğu kategoride bile %75'ın üzerinde boş olan sütunları bul
silinecek_sutunlar = minimum_bosluk_oranlari[minimum_bosluk_oranlari > 75].index.tolist()

print(f"Toplam {len(df.columns)} sütun var.")
print(f"Hiçbir kategoride %25 doluluğa bile ulaşamayan {len(silinecek_sutunlar)} çöp sütun siliniyor...")

# 4. Bu sütunları DataFrame'den tamamen uçur
df = df.drop(columns=silinecek_sutunlar)

print(f"Geriye {len(df.columns)} adet kaliteli sütun kaldı.")

Toplam 524 sütun var.
Hiçbir kategoride %25 doluluğa bile ulaşamayan 86 çöp sütun siliniyor...
Geriye 438 adet kaliteli sütun kaldı.


In [230]:
# 1. Hata vermeden (listeleri de destekleyerek) eşsiz değerleri sayan fonksiyon
def essiz_sayiyi_bul(sutun):
    try:
        return sutun.nunique()
    except TypeError:
        # Eğer sütunda liste varsa, listeleri geçici olarak tuple'a çevirip say
        return sutun.apply(lambda x: tuple(x) if isinstance(x, list) else x).nunique()

# 2. Eksik değer oranını hesapla
eksik_yuzde = df.isnull().mean() * 100

# 3. Eşsiz sayıyı hesapla
essiz_sayi = df.apply(essiz_sayiyi_bul)

# 4. Tabloyu oluştur
ozet_tablo = pd.DataFrame({
    'Eksik Değer Oranı (%)': eksik_yuzde.round(2),
    'Eşsiz Değer Sayısı': essiz_sayi
})

# En çok boşluk olanları en üstte görmek için sırala
ozet_tablo = ozet_tablo.sort_values(by='Eksik Değer Oranı (%)', ascending=False)

# Aradaki satırlar gizlenmesin diye limiti kaldır ve göster
pd.set_option('display.max_rows', None) 
ozet_tablo

,Eksik Değer Oranı (%),Eşsiz Değer Sayısı
360° Dönebilen Kablosuz Kullanım,99.54,1
IP Sınıfı,99.54,1
Enerji sınıfı (WH) (EP),99.54,1
Titanyum Emaye Kaplamalı Kazan,99.54,1
Yüksek Basınç Emniyet Valfi,99.54,1
Tank Materyali,99.54,1
Damacana Yeri,99.54,2
Soğuk su tank kapasitesi (L),99.54,2
Şok Buhar Özelliği,99.54,1
Buharsız Ütüleme,99.54,1


In [231]:
tek_degerli_rapor = []

for col in df.columns:
    try:
        # Boş olmayan verileri al
        gecerli_veri = df[col].dropna()
        
        # Eğer sadece 1 çeşit değer varsa
        if gecerli_veri.nunique() == 1:
            tek_deger = gecerli_veri.iloc[0] # O tek değeri yakala
            doluluk = (len(gecerli_veri) / len(df)) * 100
            tek_degerli_rapor.append({
                'Sütun Adı': col, 
                'İçindeki Tek Değer': tek_deger,
                'Doluluk Oranı (%)': round(doluluk, 2)
            })
            
    except TypeError:
        # Liste barındıran sütunlar için hata yakalama
        gecerli_veri = df[col].dropna()
        if gecerli_veri.apply(lambda x: tuple(x) if isinstance(x, list) else x).nunique() == 1:
            tek_deger = gecerli_veri.iloc[0]
            doluluk = (len(gecerli_veri) / len(df)) * 100
            tek_degerli_rapor.append({
                'Sütun Adı': col, 
                'İçindeki Tek Değer': str(tek_deger), # Listeyi metin gibi göster
                'Doluluk Oranı (%)': round(doluluk, 2)
            })

# Raporu DataFrame'e çevir ve doluluk oranına göre sırala
df_rapor = pd.DataFrame(tek_degerli_rapor)
df_rapor = df_rapor.sort_values(by='Doluluk Oranı (%)', ascending=False).reset_index(drop=True)

# Limitleri kaldırıp tabloyu gösterelim
pd.set_option('display.max_rows', None)
df_rapor

,Sütun Adı,İçindeki Tek Değer,Doluluk Oranı (%)
0,Wi-Fi,Var,14.22
1,2.Arka Kamera,Var,13.37
2,NFC,Var,12.45
3,Ön Kamera Flaş,Var,10.07
4,Görüntülü Konuşma,Var,9.68
5,GPS,Var,8.92
6,Ses Kayıt,Var,6.92
7,Yüz Haritalama,Var,6.69
8,Çift Hatlı,Var,5.15
9,Çocuk Kilidi,Var,5.00


In [232]:
silinecek_sutunlar = []
donusturulen_sutunlar = []

for col in df.columns:
    try:
        # Sadece eşsiz değeri 1 olan sütunlarla ilgileniyoruz
        gecerli_veri = df[col].dropna()
        if gecerli_veri.nunique() == 1:
            # O tek değeri al ve küçük harfe çevir (küçük/büyük harf karmaşasını önlemek için)
            tek_deger = str(gecerli_veri.iloc[0]).strip().lower()
            
            # KURAL 1: İstisna (Gaz Tipi -> Doğalgaz)
            if col == 'Gaz Tipi' and tek_deger == 'doğalgaz':
                df = df.rename(columns={'Gaz Tipi': 'Doğalgaz'})
                # Dolu olanları 1, boş (NaN) olanları 0 yap
                df['Doğalgaz'] = df['Doğalgaz'].notnull().astype(int)
                donusturulen_sutunlar.append('Doğalgaz (Eski adı: Gaz Tipi)')
                
            # KURAL 2: İçinde 'var' geçiyorsa tut ve 1/0 yap
            elif 'var' in tek_deger:
                df[col] = df[col].notnull().astype(int)
                donusturulen_sutunlar.append(col)
                
            # KURAL 3: İçinde 'var' geçmiyorsa ve istisna değilse silinecekler listesine at
            else:
                silinecek_sutunlar.append(col)
                
    except TypeError:
        # Liste içeren sütunlarda da aynı mantığı işlet
        gecerli_veri = df[col].dropna()
        if gecerli_veri.apply(lambda x: tuple(x) if isinstance(x, list) else x).nunique() == 1:
            tek_deger = str(gecerli_veri.iloc[0]).strip().lower()
            if 'var' in tek_deger:
                df[col] = df[col].notnull().astype(int)
                donusturulen_sutunlar.append(col)
            else:
                silinecek_sutunlar.append(col)

# 4. ADIM: İçinde "var" geçmeyen tek değerli çöp sütunları sil
df = df.drop(columns=silinecek_sutunlar)

print(f"Toplam {len(donusturulen_sutunlar)} adet 'Var' içeren sütun 1/0 formatına dönüştürülüp kurtarıldı!")
print(f"İçinde 'Var' geçmeyen {len(silinecek_sutunlar)} adet gereksiz sütun silindi.")
print("-" * 40)
print(f"Güncel Sütun Sayısı: {len(df.columns)}")

Toplam 152 adet 'Var' içeren sütun 1/0 formatına dönüştürülüp kurtarıldı!
İçinde 'Var' geçmeyen 44 adet gereksiz sütun silindi.
----------------------------------------
Güncel Sütun Sayısı: 394


In [233]:
# verbose=True ve show_counts=True parametreleri gizlemeyi önler ve tüm detayları verir
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 394 columns):
 #    Column                                                         Non-Null Count  Dtype  
---   ------                                                         --------------  -----  
 0    urun_linki                                                     1301 non-null   str    
 1    urun_adi                                                       1301 non-null   str    
 2    fiyat                                                          1176 non-null   float64
 3    enerji_sinifi                                                  270 non-null    str    
 4    favori_sayisi                                                  1237 non-null   Int64  
 5    Product_Group                                                  1301 non-null   str    
 6    Main_Category                                                  1301 non-null   str    
 7    Subcategory                                                 

In [234]:
# Renk
renk_gruplari = {
    'siyah': ['siyah', 'antrasit', 'grafit', 'piyano', 'midnight', 'dark', 'koyu', 'cosmic', 'black'],
    'gri': ['gri', 'i̇noks', 'inoks', 'gümüş', 'metalik', 'manhattan', 'platinum', 'glass', 'gray', 'grey', 'silver'],
    'mavi': ['mavi', 'buz', 'cool', 'cloud', 'arctic', 'starlight', 'quiet', 'lacivert', 'blue'],
    'pembe': ['pembe', 'rose', 'cherry', 'lilac', 'blossom', 'menekşe', 'pink'],
    'beyaz': ['beyaz', 'i̇pek', 'ipek', 'doğal', 'harmony', 'şeffaf', 'white'],
    'kırmızı': ['kırmızı', 'red'],
    'sarı': ['sarı', 'yellow', 'altın', 'gold'],
    'yeşil': ['yeşil', 'green'],
    'turuncu': ['turuncu', 'orange'],
    'bej': ['bej', 'beige'],
    'mor': ['mor', 'purple'],
    'kahverengi': ['kahverengi', 'brown']
}

# 1. ADIM: İsminde 'renk' veya 'reng' geçen tüm sütunları otomatik bul
renk_sutunlari = [col for col in df.columns if 'renk' in col.lower() or 'reng' in col.lower()]

print(f"Tespit edilen renk sütunları ({len(renk_sutunlari)} adet):\n{renk_sutunlari}\n")

# 2. ADIM: Hatalı "kolay" değerlerini NaN (boş) yap
# Regex ile büyük/küçük harf veya etrafındaki boşluklar fark etmeksizin "kolay" değerini yakalarız
for col in renk_sutunlari:
    df[col] = df[col].replace(r'(?i)^\s*kolay\s*$', np.nan, regex=True)

# 3. ADIM: Her satır için bu sütunlardaki metinleri yan yana birleştirip küçük harfe çevir
kombine_renk_metni = df[renk_sutunlari].fillna('').astype(str).agg(' '.join, axis=1).str.lower()

# 4. ADIM: Renk sözlüğünü dön ve makinenin anlayacağı 1/0 sütunlarını yarat
olusturulan_renk_sutunlari = []

for ana_renk, kelimeler in renk_gruplari.items():
    pattern = '|'.join(kelimeler)
    yeni_sutun_adi = f"Renk_{ana_renk.capitalize()}"
    olusturulan_renk_sutunlari.append(yeni_sutun_adi)
    
    # Kelimelerden biri geçiyorsa 1, geçmiyorsa 0 ata
    df[yeni_sutun_adi] = kombine_renk_metni.str.contains(pattern, case=False, na=False).astype(int)

# 5. ADIM: Dağınıklık yaratan eski renk sütunlarını DataFrame'den tamamen sil
df = df.drop(columns=renk_sutunlari)

print("İşlem Başarılı! 'Kolay' hataları temizlendi, renkler 1/0 olarak gruplandı ve eski sütunlar silindi.")
print("-" * 50)
print(f"Oluşturulan Yeni Sütunlar:\n{olusturulan_renk_sutunlari}\n")
print(f"Güncel Sütun Sayısı: {len(df.columns)}")

Tespit edilen renk sütunları (8 adet):
['Ürün Rengi', 'Cam Dekor Rengi', 'Renk', 'Kanat Rengi', 'Gövde Rengi', 'Cezve Rengi', 'Ürün rengi', 'Gövde rengi']

İşlem Başarılı! 'Kolay' hataları temizlendi, renkler 1/0 olarak gruplandı ve eski sütunlar silindi.
--------------------------------------------------
Oluşturulan Yeni Sütunlar:
['Renk_Siyah', 'Renk_Gri', 'Renk_Mavi', 'Renk_Pembe', 'Renk_Beyaz', 'Renk_Kırmızı', 'Renk_Sarı', 'Renk_Yeşil', 'Renk_Turuncu', 'Renk_Bej', 'Renk_Mor', 'Renk_Kahverengi']

Güncel Sütun Sayısı: 398


In [235]:
# Enerji Sınıfı 
# 1. ÇELİŞENLER: İki sütun da dolu ama değerler BİRBİRİNDEN FARKLI MI?
celisenler = df[(df['enerji_sinifi'].notnull()) & (df['Enerji Sınıfı'].notnull()) & (df['enerji_sinifi'] != df['Enerji Sınıfı'])]

print("1. ÇELİŞEN DEĞERLER (İki sütun da dolu ama içindeki bilgiler farklı)")
print("-" * 60)
if len(celisenler) > 0:
    print(f"Dikkat! {len(celisenler)} satırda iki sütun birbiriyle çelişiyor. İşte o satırlar:")
    display(celisenler[['enerji_sinifi', 'Enerji Sınıfı']])
else:
    print("Harika! İki sütunun da dolu olduğu yerlerde hiçbir çelişki (farklılık) yok.")


# 2. EKSİĞİ TAMAMLAYANLAR: Ana sütunda olmayıp, diğerinde olanlar
farkli_olanlar = df[df['enerji_sinifi'].isnull() & df['Enerji Sınıfı'].notnull()]

print("\n2. EKLENECEK FARKLI DEĞERLER (Ana sütunda yok, yedek sütunda var)")
print("-" * 60)
if len(farkli_olanlar) > 0:
    print(f"Ana sütuna aktarılacak toplam {len(farkli_olanlar)} adet veri var. İşte tablosu:")
    # Çok uzun olmasın diye ilk 15 tanesini gösterelim (istersen .head(15) kısmını silebilirsin)
    display(farkli_olanlar[['enerji_sinifi', 'Enerji Sınıfı']].head(15))
else:
    print("Aktarılacak farklı bir veri bulunamadı.")


# 3. BİRLEŞTİRME VE TEMİZLİK İŞLEMİ
print("\n3. BİRLEŞTİRME SONUCU")
print("-" * 60)
# 'enerji_sinifi'ndeki boş (NaN) olan yerleri 'Enerji Sınıfı'ndaki değerlerle doldur
df['enerji_sinifi'] = df['enerji_sinifi'].fillna(df['Enerji Sınıfı'])

# İşi biten yedek sütunu tamamen sil
df = df.drop(columns=['Enerji Sınıfı'])

print("İşlem Başarılı! Farklı olan/eksik olan veriler ana sütuna aktarıldı ve yedek sütun silindi.")
print(f"Güncel 'enerji_sinifi' dolu satır sayısı: {df['enerji_sinifi'].notnull().sum()}")


1. ÇELİŞEN DEĞERLER (İki sütun da dolu ama içindeki bilgiler farklı)
------------------------------------------------------------
Harika! İki sütunun da dolu olduğu yerlerde hiçbir çelişki (farklılık) yok.

2. EKLENECEK FARKLI DEĞERLER (Ana sütunda yok, yedek sütunda var)
------------------------------------------------------------
Aktarılacak farklı bir veri bulunamadı.

3. BİRLEŞTİRME SONUCU
------------------------------------------------------------
İşlem Başarılı! Farklı olan/eksik olan veriler ana sütuna aktarıldı ve yedek sütun silindi.
Güncel 'enerji_sinifi' dolu satır sayısı: 270


In [236]:
# Su Tüketimi

#  Ana sütun ve yedek sütun isimlerini belirliyoruz
ana_sutun = 'Su Tüketimi'
yedek_sutun = 'Su Tüketimi (1 Çevrim)'

# 1. ÇELİŞENLER: İki sütun da dolu ama değerler BİRBİRİNDEN FARKLI MI?
celisenler = df[(df[ana_sutun].notnull()) & (df[yedek_sutun].notnull()) & (df[ana_sutun] != df[yedek_sutun])]

print("1. ÇELİŞEN DEĞERLER (İki sütun da dolu ama içindeki bilgiler farklı)")
print("-" * 60)
if len(celisenler) > 0:
    print(f"Dikkat! {len(celisenler)} satırda iki sütun birbiriyle çelişiyor. İşte o satırlar:")
    display(celisenler[[ana_sutun, yedek_sutun]])
else:
    print(f"Harika! '{ana_sutun}' ve '{yedek_sutun}' sütunlarının ikisinin de dolu olduğu yerlerde hiçbir çelişki yok.")


# 2. EKSİĞİ TAMAMLAYANLAR: Ana sütunda olmayıp, diğerinde olanlar
farkli_olanlar = df[df[ana_sutun].isnull() & df[yedek_sutun].notnull()]

print("\n2. EKLENECEK FARKLI DEĞERLER (Ana sütunda yok, yedek sütunda var)")
print("-" * 60)
if len(farkli_olanlar) > 0:
    print(f"Ana sütuna aktarılacak toplam {len(farkli_olanlar)} adet veri var. İşte tablosu:")
    display(farkli_olanlar[[ana_sutun, yedek_sutun]].head(15))
else:
    print("Aktarılacak farklı bir veri bulunamadı.")


# 3. BİRLEŞTİRME VE TEMİZLİK İŞLEMİ
print("\n3. BİRLEŞTİRME SONUCU")
print("-" * 60)
# 'Su Tüketimi'ndeki boş (NaN) olan yerleri 'Su Tüketimi (1 Çevrim)'ndeki değerlerle doldur
df[ana_sutun] = df[ana_sutun].fillna(df[yedek_sutun])

# İşi biten yedek sütunu tamamen sil
df = df.drop(columns=[yedek_sutun])

print(f"İşlem Başarılı! Veriler '{ana_sutun}' sütununda birleştirildi ve '{yedek_sutun}' sütunu silindi.")
print(f"Güncel '{ana_sutun}' dolu satır sayısı: {df[ana_sutun].notnull().sum()}")

1. ÇELİŞEN DEĞERLER (İki sütun da dolu ama içindeki bilgiler farklı)
------------------------------------------------------------
Harika! 'Su Tüketimi' ve 'Su Tüketimi (1 Çevrim)' sütunlarının ikisinin de dolu olduğu yerlerde hiçbir çelişki yok.

2. EKLENECEK FARKLI DEĞERLER (Ana sütunda yok, yedek sütunda var)
------------------------------------------------------------
Ana sütuna aktarılacak toplam 19 adet veri var. İşte tablosu:


,Su Tüketimi,Su Tüketimi (1 Çevrim)
79,NaN,51 L
80,NaN,49 L
81,NaN,51 L
82,NaN,44 L
83,NaN,44 L
84,NaN,49 L
85,NaN,56 L
86,NaN,52 L
87,NaN,49 L
88,NaN,40 L



3. BİRLEŞTİRME SONUCU
------------------------------------------------------------
İşlem Başarılı! Veriler 'Su Tüketimi' sütununda birleştirildi ve 'Su Tüketimi (1 Çevrim)' sütunu silindi.
Güncel 'Su Tüketimi' dolu satır sayısı: 52


In [237]:
# Su Hazanesi - Su Tankı

def kapasite_duzelt(deger):
    if pd.isna(deger):
        return deger
    
    # Sayıyı ve birimi ayırmak için regex (Örn: "350 mL" içinden 350'yi ve mL'yi alır)
    deger_str = str(deger).lower().replace(',', '.')
    sayi_ara = re.search(r"(\d+\.?\d*)", deger_str)
    
    if sayi_ara:
        sayi = float(sayi_ara.group(1))
        # Eğer birim mL ise 1000'e bölerek Litre'ye çevir
        if 'ml' in deger_str:
            return sayi / 1000
        # Diğer durumlarda (Litre varsayıyoruz) sayıyı olduğu gibi bırak
        return sayi
    return np.nan

# 1. ADIM: Mevcut 'Su Tankı Kapasitesi'ni sayısallaştır ve mL -> L dönüşümü yap
df['Su Tankı Kapasitesi'] = df['Su Tankı Kapasitesi'].apply(kapasite_duzelt)

# 2. ADIM: Diğer sütun olan 'Soğuk su tank kapasitesi (L)' için de aynısını yap ve birleştir
if 'Soğuk su tank kapasitesi (L)' in df.columns:
    df['Soğuk su tank kapasitesi (L)'] = df['Soğuk su tank kapasitesi (L)'].apply(kapasite_duzelt)
    
    # Ana sütunu yedek sütunla doldur
    df['Su Tankı Kapasitesi'] = df['Su Tankı Kapasitesi'].fillna(df['Soğuk su tank kapasitesi (L)'])
    
    # İşi biten yedek sütunu sil
    df = df.drop(columns=['Soğuk su tank kapasitesi (L)'])

print("Birimler Standardize Edildi!")
print("-" * 40)
print(f"Su Tankı Kapasitesi sütunu artık 'Litre' cinsinden sayısal değerler içeriyor.")
print(f"Örnek Değerler: {df['Su Tankı Kapasitesi'].dropna().unique()[:5]}")
print(f"Toplam Dolu Satır: {df['Su Tankı Kapasitesi'].notnull().sum()}")

Birimler Standardize Edildi!
----------------------------------------
Su Tankı Kapasitesi sütunu artık 'Litre' cinsinden sayısal değerler içeriyor.
Örnek Değerler: [ 3.   3.6 44.   8.   6. ]
Toplam Dolu Satır: 35


In [238]:
# Ağırlık

def agirlik_temizle_ve_kg_yap(deger):
    if pd.isna(deger):
        return deger
    
    # Metni temizle: küçük harf yap, virgülü noktaya çevir
    deger_str = str(deger).lower().replace(',', '.')
    
    # Düzenli ifade (Regex) ile sayısal kısmı çek
    sayi_ara = re.search(r"(\d+\.?\d*)", deger_str)
    
    if sayi_ara:
        sayi = float(sayi_ara.group(1))
        
        # Birim kontrolü: Eğer gram ise 1000'e bölerek kg'a çevir
        if 'gr' in deger_str or 'gram' in deger_str:
            return sayi / 1000
        
        # Diğer durumlarda (kg veya birimsiz ise) kg kabul et
        return sayi
    return np.nan

# 1. ADIM: Ağırlıkla ilgili tüm sütunları tespit et
agirlik_sutunlari = [col for col in df.columns if 'ağırlık' in col.lower()]

# 2. ADIM: Her bir sütuna temizleme ve kg dönüşüm fonksiyonunu uygula
for col in agirlik_sutunlari:
    df[col] = df[col].apply(agirlik_temizle_ve_kg_yap)

# 3. ADIM: Tek bir standart isimli sütunda (Agirlik (kg)) birleştir
# Önce boş bir float sütun oluşturuyoruz
df['Agirlik (kg)'] = np.nan

# Hiyerarşik birleştirme (Önce net ağırlıkları, sonra paketliyi al)
oncelik_sirasi = ['Ağırlık: Paketsiz', 'Ağırlık (gr)', 'Ağırlık', 'Ağırlık: Paketli']

for sutun in oncelik_sirasi:
    if sutun in df.columns:
        df['Agirlik (kg)'] = df['Agirlik (kg)'].fillna(df[sutun])

# 4. ADIM: Veri tipini kesin olarak float yap ve eski sütunları sil
df['Agirlik (kg)'] = df['Agirlik (kg)'].astype(float)
df = df.drop(columns=agirlik_sutunlari, errors='ignore')

print("Ağırlık Verileri Standardize Edildi!")
print("-" * 50)
print(f"Yeni Sütun Adı: Agirlik (kg)")
print(f"Veri Tipi: {df['Agirlik (kg)'].dtype}")
print(f"Dolu Satır Sayısı: {df['Agirlik (kg)'].notnull().sum()}")
print("\nÖrnek Değerler (Kg cinsinden):")
print(df['Agirlik (kg)'].dropna().head())

Ağırlık Verileri Standardize Edildi!
--------------------------------------------------
Yeni Sütun Adı: Agirlik (kg)
Veri Tipi: float64
Dolu Satır Sayısı: 624

Örnek Değerler (Kg cinsinden):
0   115.00
1   105.00
2   105.00
3   105.00
4    87.50
Name: Agirlik (kg), dtype: float64


In [239]:
# Şarj Süresi

def sure_temizle_ve_dakika_yap(deger):
    if pd.isna(deger):
        return deger
    
    deger_str = str(deger).lower().replace(',', '.')
    
    # Sayıları bul (Örn: "3.5", "4-5" gibi durumlarda ilk sayıyı veya ortalamayı alabiliriz)
    # Burada ilk bulunan sayıyı alıyoruz
    sayilar = re.findall(r"(\d+\.?\d*)", deger_str)
    
    if sayilar:
        sayi = float(sayilar[0])
        
        # Birim kontrolü
        # Eğer 'saat' geçiyorsa 60 ile çarp (dakikaya çevir)
        if 'saat' in deger_str or 'sa' in deger_str or 'hr' in deger_str:
            return sayi * 60
        
        # Zaten dakika ise (dk, daki, min) olduğu gibi bırak
        return sayi
    
    return np.nan

# 1. ADIM: İlgili sütunları tespit et
sarj_sutunlari = ['Şarj Olma Süresi', 'Şarj süresi']

# 2. ADIM: Temizleme ve dakika dönüşümünü uygula
for col in sarj_sutunlari:
    if col in df.columns:
        df[col] = df[col].apply(sure_temizle_ve_dakika_yap)

# 3. ADIM: 'Sarj_Suresi_dk' isminde tek bir sütunda birleştir
df['Sarj_Suresi_dk'] = np.nan

if 'Şarj Olma Süresi' in df.columns:
    df['Sarj_Suresi_dk'] = df['Sarj_Suresi_dk'].fillna(df['Şarj Olma Süresi'])

if 'Şarj süresi' in df.columns:
    df['Sarj_Suresi_dk'] = df['Sarj_Suresi_dk'].fillna(df['Şarj süresi'])

# 4. ADIM: Eski sütunları sil
df = df.drop(columns=[col for col in sarj_sutunlari if col in df.columns], errors='ignore')

print("Şarj Süreleri Standardize Edildi!")
print("-" * 50)
print(f"Yeni Sütun Adı: Sarj_Suresi_dk")
print(f"Dolu Satır Sayısı: {df['Sarj_Suresi_dk'].notnull().sum()}")
print("\nÖrnek Değerler (Dakika cinsinden):")
print(df['Sarj_Suresi_dk'].dropna().unique()[:5])

Şarj Süreleri Standardize Edildi!
--------------------------------------------------
Yeni Sütun Adı: Sarj_Suresi_dk
Dolu Satır Sayısı: 32

Örnek Değerler (Dakika cinsinden):
[330. 180. 240. 120.  60.]


In [243]:
import re
import pandas as pd
import numpy as np

# 1. ADIM: GENEL BOOLEAN (EVET/VAR) DÖNÜŞÜMÜ (Hata Onarılmış Sürüm)
def boolean_cevir(val):
    # Eğer değer bir liste veya dizi ise, işlem yapmadan olduğu gibi bırak
    if isinstance(val, (list, np.ndarray, pd.Series, dict)): 
        return val
    
    # Boş değer kontrolü (Scalar değerler için)
    if pd.isna(val): 
        return val
        
    s = str(val).lower().strip()
    if s in ['evet', 'var', 'mevcut', 'yes', 'true', '1.0', '1']: 
        return 1
    if s in ['hayır', 'yok', 'mevcut değil', 'no', 'false', '0.0', '0']: 
        return 0
    return val

# Sadece metin sütunlarını seçip hatasız şekilde uygulayalım
obj_cols = df.select_dtypes(include=['object']).columns
for col in obj_cols:
    df[col] = df[col].apply(boolean_cevir)

# 2. ADIM: GELİŞMİŞ SAYI TEMİZLEYİCİ (BİRİM ODAKLI)
def ultra_temizleyici(deger):
    # Liste veya dizi gelirse temizleme işlemini pas geç (hata önleme)
    if isinstance(deger, (list, np.ndarray, dict)) or pd.isna(deger) or deger == "":
        return deger
    
    # Eğer zaten sayıysa, olduğu gibi bırak
    if isinstance(deger, (int, float)): 
        return float(deger)
    
    txt = str(deger).lower().replace(',', '.')
    
    # Regex: Birimin yanındaki sayıyı yakala
    match = re.search(r"(\d+\.?\d*)\s*(w|watt|kw|l|lt|litre|ml|gb|mb|mah|btu|kg|gr|gram|dk|dakika|saat|sa|cm|mm|db|kwh)", txt)
    
    if match:
        sayi = float(match.group(1))
        birim = match.group(2)
        if birim == 'kw': return sayi * 1000
        if birim == 'ml': return sayi / 1000
        if birim == 'mb': return sayi / 1024
        if birim in ['gr', 'gram']: return sayi / 1000
        if birim in ['saat', 'sa']: return sayi * 60
        if birim == 'mm': return sayi / 10
        return sayi
    
    # Sayıları bul ve en büyüğünü al
    sayilar = re.findall(r"(\d+\.?\d*)", txt)
    if sayilar:
        return max([float(s) for s in sayilar])
    return np.nan

# 3. ADIM: SAYISAL SÜTUNLARI TEMİZLEME
temizlenecek_sayisallar = ['Genişlik', 'Yükseklik', 'Derinlik', 'Ses Seviyesi', 'Yıllık Enerji Tüketimi (kWh)', 'fiyat']
for col in temizlenecek_sayisallar:
    if col in df.columns:
        df[col] = df[col].apply(ultra_temizleyici)

# 4. ADIM: GRUP BİRLEŞTİRME
mapping = {
    'Guc (W)': ['Lamba Gücü (W)', 'Güç (W) (9)', 'Motor Gücü', 'Isıtmada Çekilen Enerji (W)', 'Çıkış Gücü', 'Giriş Gücü'],
    'Agirlik (kg)': ['Ağırlık: Paketsiz', 'Ağırlık (gr)', 'Ağırlık: Paketli', 'Günlük Dondurma Kapasitesi (kg/Gün)'],
    'Sarj_Suresi_dk': ['Şarj Olma Süresi', 'Şarj süresi', 'Kablosuz Çalışma Süresi (Düşük Güç)', 'Kablosuz çalışma süresi(Orta güç)'],
    'Su_Kapasitesi (L)': ['Toplam Hacim (L)', 'Soğutucu Bölme Hacmi (L)', 'Dondurucu Bölme Hacmi (L)', 'Su Tankı Kapasitesi (L)', 'Su Haznesi'],
    'RAM (GB)': ['RAM Kapasitesi', 'Bellek'],
    'Depolama (GB)': ['SSD Kapasitesi', 'Dahili Depolama Kapasitesi', 'Harddisk Kapasitesi', 'Dahili Hafıza'],
    'Batarya (mAh)': ['Pil Kapasitesi', 'Batarya kapasitesi', 'Pil']
}

for yeni, eskiler in mapping.items():
    if yeni not in df.columns:
        df[yeni] = np.nan
    for eski in eskiler:
        if eski in df.columns:
            temiz_veri = df[eski].apply(ultra_temizleyici)
            df[yeni] = df[yeni].combine_first(temiz_veri)

# 5. ADIM: BOYUT PARÇALAMA
if 'Boyut (cm) (GxYxD)' in df.columns:
    boyut_split = df['Boyut (cm) (GxYxD)'].astype(str).str.extract(r'(\d+\.?\d*)\s*[xX*]\s*(\d+\.?\d*)\s*[xX*]\s*(\d+\.?\d*)')
    df['Genişlik'] = df['Genişlik'].combine_first(pd.to_numeric(boyut_split[0], errors='coerce'))
    df['Yükseklik'] = df['Yükseklik'].combine_first(pd.to_numeric(boyut_split[1], errors='coerce'))
    df['Derinlik'] = df['Derinlik'].combine_first(pd.to_numeric(boyut_split[2], errors='coerce'))

# 6. ADIM: TEMİZLİK VE TİP ZORLAMASI
silinecekler = [item for sublist in mapping.values() for item in sublist] + ['Boyut (cm) (GxYxD)']
df.drop(columns=[c for c in silinecekler if c in df.columns], errors='ignore', inplace=True)

# Zorunlu float dönüşümü
hedef_sutunlar = list(mapping.keys()) + temizlenecek_sayisallar
for col in hedef_sutunlar:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Hata giderildi ve süreç tamamlandı! Sütunlar artık sayısal ve dolu.")

Hata giderildi ve süreç tamamlandı! Sütunlar artık sayısal ve dolu.


In [244]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 375 columns):
 #    Column                                                         Non-Null Count  Dtype  
---   ------                                                         --------------  -----  
 0    urun_linki                                                     1301 non-null   str    
 1    urun_adi                                                       1301 non-null   str    
 2    fiyat                                                          1176 non-null   float64
 3    enerji_sinifi                                                  270 non-null    str    
 4    favori_sayisi                                                  1237 non-null   Int64  
 5    Product_Group                                                  1301 non-null   str    
 6    Main_Category                                                  1301 non-null   str    
 7    Subcategory                                                 

In [245]:
# --- 1. SES SEVİYESİ VE DEVİR TEMİZLİĞİ ---
def sadece_sayi_cek(val):
    if pd.isna(val) or isinstance(val, (int, float)): return val
    num = re.search(r"(\d+)", str(val))
    return float(num.group(1)) if num else np.nan

# Ses Seviyelerini birleştir
if 'Ses Gücü Seviyesi (dBA)' in df.columns:
    df['Ses Seviyesi'] = df['Ses Seviyesi'].fillna(df['Ses Gücü Seviyesi (dBA)'].apply(sadece_sayi_cek))
    df.drop(columns=['Ses Gücü Seviyesi (dBA)'], inplace=True)

# Sıkma Devrini sayısallaştır
if 'Maksimum Sıkma Devri' in df.columns:
    df['Sikma_Devri'] = df['Maksimum Sıkma Devri'].apply(sadece_sayi_cek)
    df.drop(columns=['Maksimum Sıkma Devri'], inplace=True)

# --- 2. GERİDE KALAN HACİM SÜTUNLARINI STANDARTLAŞTIRMA ---
# Tambur, Fırın, Toz Haznesi gibi hacimleri ultra_temizleyici ile float yap
hacim_adaylari = ['Tambur Hacmi (L)', 'Fırın Hacmi', 'Kurutma Kapasitesi', 
                  'Pişirme Kapasitesi', 'İç Hazne Kapasitesi', 'Toz Haznesi Kapasitesi']

for col in hacim_adaylari:
    if col in df.columns:
        df[col] = df[col].apply(ultra_temizleyici)

# --- 3. ÇOK SEYREK SÜTUNLARI SİLME ---
# 10'dan az veri içeren sütunları temizleyerek gürültüyü azaltıyoruz
limit = 10
seyrek_cols = [col for col in df.columns if df[col].count() < limit]
# Önemli olabilecek kategori sütunlarını koruyalım
korunacaklar = ['urun_linki', 'urun_adi', 'fiyat']
seyrek_cols = [c for c in seyrek_cols if c not in korunacaklar]

df.drop(columns=seyrek_cols, inplace=True)

# --- 4. TİP DÜZELTME (OBJECT -> STR/INT) ---
# Hala object görünen ama aslında string olanları netleştirelim
obj_to_str = df.select_dtypes(include=['object']).columns
for col in obj_to_str:
    # Liste içermeyenleri str yapalım
    if not df[col].apply(lambda x: isinstance(x, list)).any():
        df[col] = df[col].astype(str).replace('nan', np.nan)

print(f"Final Dokunuş Tamamlandı! Kalan Sütun Sayısı: {len(df.columns)}")

Final Dokunuş Tamamlandı! Kalan Sütun Sayısı: 340


In [246]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 340 columns):
 #    Column                                                         Non-Null Count  Dtype  
---   ------                                                         --------------  -----  
 0    urun_linki                                                     1301 non-null   str    
 1    urun_adi                                                       1301 non-null   str    
 2    fiyat                                                          1176 non-null   float64
 3    enerji_sinifi                                                  270 non-null    str    
 4    favori_sayisi                                                  1237 non-null   Int64  
 5    Product_Group                                                  1301 non-null   str    
 6    Main_Category                                                  1301 non-null   str    
 7    Subcategory                                                 

In [247]:
# Adım 1: Güç Sütunlarını Nihai Olarak Birleştirme
# 114 (Güç) ve 109 (kW) sütunlarını temizleyip ana sütuna aktaralım
guc_ek_adaylar = ['Güç', 'Toplam Elektrik Gücü (kW)', 'Güç (W)', 'Güç (W) (9)']

for col in guc_ek_adaylar:
    if col in df.columns:
        # Daha önce yazdığımız ultra_temizleyici'yi kullanarak sayıya çeviriyoruz
        cleaned = df[col].apply(ultra_temizleyici)
        df['Guc (W)'] = df['Guc (W)'].combine_first(cleaned)

# Adım 2: Enerji ve Su Tüketimini Sayısallaştırma
tuketim_cols = ['Günlük Enerji Tüketimi (kwh/Gün)', 'Enerji Tüketimi (100 Çevrim)', 'Su Tüketimi']
for col in tuketim_cols:
    if col in df.columns:
        df[col] = df[col].apply(ultra_temizleyici)

# Adım 3: Kalan Teknik Özellikleri Temizleme
teknik_cols = ['Maksimum Sıkma Devri', 'Kapasite', 'Toz Torbası Kapasitesi']
for col in teknik_cols:
    if col in df.columns:
        df[col] = df[col].apply(ultra_temizleyici)

# Adım 4: Gereksizleşen Ara Sütunları Silme
# Artık Guc (W) içinde toplandıkları için orijinallerini silebiliriz
df.drop(columns=[c for c in guc_ek_adaylar if c in df.columns], errors='ignore', inplace=True)

# Adım 5: Çok Seyrek Sütunları Budama
# Bir satırda 20'den az (yaklaşık %1.5 doluluk) veri varsa o sütunu silelim
limit = 20
seyrek_sutunlar = [col for col in df.columns if df[col].count() < limit]
# Fiyat gibi hedef değişkenleri koru
korunacaklar = ['urun_linki', 'urun_adi', 'fiyat', 'Main_Category', 'Subcategory']
seyrek_sutunlar = [c for c in seyrek_sutunlar if c not in korunacaklar]

df.drop(columns=seyrek_sutunlar, inplace=True)

# Adım 6: Final Tip Zorlaması (Object -> String)
# Kalan metin sütunlarını netleştirelim
obj_cols = df.select_dtypes(include=['object']).columns
for col in obj_cols:
    df[col] = df[col].astype(str).replace('nan', np.nan)

print(f"Düzenleme Tamamlandı. Kalan Sütun Sayısı: {len(df.columns)}")

Düzenleme Tamamlandı. Kalan Sütun Sayısı: 285


In [249]:
df.info(verbose=True, show_counts=True  )

<class 'pandas.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 285 columns):
 #    Column                                                         Non-Null Count  Dtype  
---   ------                                                         --------------  -----  
 0    urun_linki                                                     1301 non-null   str    
 1    urun_adi                                                       1301 non-null   str    
 2    fiyat                                                          1176 non-null   float64
 3    enerji_sinifi                                                  270 non-null    str    
 4    favori_sayisi                                                  1237 non-null   Int64  
 5    Product_Group                                                  1301 non-null   str    
 6    Main_Category                                                  1301 non-null   str    
 7    Subcategory                                                 

In [252]:


# --- 1. GÜÇ VE PERFORMANS HAVUZU ---
# Kaçak güç sütunlarını ana Guc (W) sütununa aktarıyoruz
guc_ek_sutunlar = ['Güç', 'Toplam Elektrik Gücü (kW)', 'Motor Gücü', 'Isıtma Gücü', 'Güç (W)']

for col in guc_ek_sutunlar:
    if col in df.columns:
        # ultra_temizleyici fonksiyonunu (önceki adımda tanımladığımız) kullanıyoruz
        cleaned_val = df[col].apply(ultra_temizleyici)
        df['Guc (W)'] = df['Guc (W)'].combine_first(cleaned_val)

# --- 2. SES SEVİYESİ BİRLEŞTİRME ---
if 'Minimum Ses Seviyesi' in df.columns:
    cleaned_min_ses = df['Minimum Ses Seviyesi'].apply(ultra_temizleyici)
    df['Ses Seviyesi'] = df['Ses Seviyesi'].combine_first(cleaned_min_ses)

# --- 3. TEKNİK ÖZELLİKLERİ SAYISALLAŞTIRMA (STR -> FLOAT) ---
# Hala str duran ama sayısal olanları zorla dönüştürüyoruz
zorunlu_sayisallar = [
    'Yıllık Enerji Tüketimi (kWh)', 'Günlük Enerji Tüketimi (kwh/Gün)', 
    'Su Tüketimi', 'Fincan Kapasitesi', 'Toz Torbası Kapasitesi'
]

for col in zorunlu_sayisallar:
    if col in df.columns:
        df[col] = df[col].apply(ultra_temizleyici)

# --- 4. SEYREK SÜTUNLARI BUDAMA (%1.5 ALTINDAKİLER) ---
limit = 20
seyrek_sutunlar = [col for col in df.columns if df[col].count() < limit]
# Kritik kategori ve hedef sütunlarını koru
korunacaklar = ['urun_linki', 'urun_adi', 'fiyat', 'Main_Category', 'Subcategory', 'Product_Group']
seyrek_sutunlar = [c for c in seyrek_sutunlar if c not in korunacaklar]

df.drop(columns=seyrek_sutunlar, inplace=True)

# --- 5. GEREKSİZLEŞEN ARA SÜTUNLARI SİLME ---
df.drop(columns=[c for c in guc_ek_sutunlar if c in df.columns], errors='ignore', inplace=True)
if 'Minimum Ses Seviyesi' in df.columns: df.drop(columns=['Minimum Ses Seviyesi'], inplace=True)

# --- 6. KATEGORİK SÜTUNLARI STANDARTLAŞTIRMA ---
# Kalan metin sütunlarını temiz str tipine çeviriyoruz
obj_cols = df.select_dtypes(include=['object']).columns
for col in obj_cols:
    df[col] = df[col].astype(str).replace('nan', np.nan).str.strip()

print(f"Optimizasyon Tamamlandı! Yeni Sütun Sayısı: {len(df.columns)}")

Optimizasyon Tamamlandı! Yeni Sütun Sayısı: 284


In [253]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

df.sample(10, random_state=42)

,urun_linki,urun_adi,fiyat,enerji_sinifi,favori_sayisi,Product_Group,Main_Category,Subcategory,Derinlik,Genişlik,Yükseklik,Sıcaklık Yükselme Süresi (Saat),Dondurucu Yeri,Elektronik Gösterge,Kontrol Sistemi,Tatil Modu,ProSmart™ Inverter Kompresör,Aydınlatma Tipi,Kapı Yönü Değiştirme,İklim Sınıfı,Ses Seviyesi,Ses Seviyesi Sınıfı,Soğutma Sistemi,Soğutucu Bölme Fanı,EverFresh+ Teknolojisi,Kahvaltılık Çekmecesi,Dondurucu Çekmece Sayısı,Buzluk Tipi,Dondurucu Bölme Aydınlatma Tipi,Sebzelik Adedi,Yıllık Enerji Tüketimi (kWh),Günlük Enerji Tüketimi (kwh/Gün),Günlük enerji tüketimi 16°C (kWh/24h) (EU_2021_EP),Ürün Tipi,Hızlı Soğutma,Hızlı Dondurma Bölmesi,Kapasite,IronFast,Çocuk Kilidi,Deterjan Sistemi,Buhar,Motor Tipi,Tambur Deseni,Elektronik Su Kontrol Sistemi,Gösterge Tipi,Kalan Zaman Göstergesi,Su Taşma Emniyet Sistemi,Dengesiz Yük Kontrolü,Su Kesik İkazı,Programlar,Dinamik Rezistans,Bağlantı,Fonksiyonlar,Ekran Tipi,Paslanmaz Çelik İç Gövde,Otomatik Kapı Açma Özelliği,Bardak Koruma Sistemi,Yarım Yük Yıkama,Yüksekliği Ayarlanabilir Fincan Rafları,Fincan Raf Sayısı,Çatal Kaşık Sepeti,Enerji Tüketimi,Su Tüketimi,Parlatıcı Göstergesi,Tuz Göstergesi,Su Güvenlik Sistemi,Kir Sensörü,Dokunmatik Tuş,Direkt Su Tahliyesi,Su Tankı Dolu Uyarısı,Kondenser Temizleme Uyarısı,Fıltre Temizleme Uyarısı,Ürün Serisi,Fırın Hacmi,Fırın İçi Aydınlatma,Elektrikli Izgara,Ön Kapak Cam Sayısı,Elektroturbo (Fan Destekli Pişirme),Saat Tipi,Fan ve Alt Isıtıcı,Alt Isıtıcı,Fan Destekli Isıtıcı,Küçük Izgara,Kolayca Sökülebilir ve Temizlenebilir Komple İç Cam,Buharlı Pişirme,Doğalgaz,Ocak Yüzeyi,Kahve Beki Adaptörü,Gaz Emniyet Sistemi,3D Pişirme,FitFry Teknolojisi,Ekran Kilidi,Ocak Tipi Ve Göz Sayısı,Ocak Tipi,Bek Şapkası Tipi,Kontrol Tipi ve Yeri,Sol Ön Bölme,Sağ Arka Bölme,Tencere Izgara Tipi,Çerçeve,Sağ Ön Bölme,Sol Arka Bölme,Zamanlayıcı,Sıcaklık Ayarı,Bulaşık Makinesinde Yıkanabilir Aksesuarlar,LCD Ekran,Aşırı Isınma Emniyeti,Davlumbaz Tipi,Maksimum Çekiş Gücü,Yağ Filtre Adedi,Montaj Yüksekliği (mm),Bulaşık Makinesinde Yıkanabilir Filtre,Kontrol Tipi,Lamba Adedi,Yoğun Emiş Gücü,Otomatik Kapanma Özelliği,Kademe Sayısı,Filtre Tipi,Filtre Doluluk Göstergesi,Her Kademe İçin Ses Güçleri (DIN/EN 60704-3 standardına göre),Hava Temizleme Özelliği,Önden Ayarlanablir Arka Ayaklar,Yüksek Bazaya Montaj,Ekran Boyutu,İşletim Sistemi,USB Bağlantısı,HDMI,Bluetooth,Arka Kamera,Ön Kamera,İşlemci,Ekran Çözünürlüğü,Mikrofon,GPS,Kulaklık Tipi,NFC,Aux Girişi,Uzaktan Kumanda,İşlemci Çekirdek Sayısı,2.Arka Kamera,Kamera Zoom,Ön Kamera Flaş,Wi-Fi,Marka,Yüz Haritalama,Çift Hatlı,Ses Kayıt,Ürün Ailesi,İşletim Sistemi Versionu,Görüntülü Konuşma,Uyumlu olduğu modeller (MPA),Hızlı Şarj desteği,Type¬C veya Micro USB kablolarıyla şarj edilebilir,Su Geçirmezlik,Soğutma Kapasitesi,Isıtma Kapasitesi (Btu/h),Yıllık Tüketim (kW) Soğuma,Yıllık Tüketim Isıtma (kW),Voltaj,Soğutucu Akışkan,Mevsimsel (SCOP) Enerji Verimi Isıtma (W/W),Mevsimsel (SEER) Enerji Verimi Soğutma (W/W),Akıllı Çalışma Sistemi,Gizli Gösterge,Uyku Mode,Dış Ünite Soğutma Çalışma Aralığı(°C),Dış Ünite Isıtma Çalışma Aralığı(°C),Özel Filtreler,Otomatik Temizleme,Otomatik Hava Yönlendirme (Yukarı-Aşağı),4 Yönlü Otomatik Hava Kontrolü,Nem Alma,Enerji Sınıfı-Isıtma,Kapasite (Isıtma),Anlık Tüketim Göstergesi,Enerji Kontrol,Sessiz Mod,Yumuşak Hava,Auto Swing,Dış Ünite (GxYxD) mm,İç Ünite (GxYxD) mm,Su Tankı Kapasitesi,Devrilme Emniyeti,Titanyum Emaye Kaplamalı Kazan,Yüksek Basınç Emniyet Valfi,Susuz Çalışma Emniyeti,Donma Emniyeti,Korozyona Karşı Koruma,Yerden Isıtmaya Uygunluk,Oda Termostatı İle Çalışma,Alev Modülasyonu,Fan Modülasyonu,Arıza Teşhis Sistemi,Emniyet Termostatı,Emniyet Ventili,Hava Emiş Emniyeti,Alev Sönme Emniyeti,3 Yollu Vana Blokaj Sistemi,Otomatik By-Pass Sistemi,Pompa Anti Blokaj Sistemi,Fincan Kapasitesi,Işıklı Uyarı,Ölçü Kaşığı,Sesli Uyarı,Cook Sense Teknolojisi (Pişmeyi Algılama),Anti Spill Teknolojisi (Taşmayı Önleme),Tek Tuşla Kontrol,Fincan Boyutu Seçimi,Su Seviye Göstergesi,Otomatik Kapanma,Sıcak Tutma Özelliği,

In [254]:

# 1. ADIM: LİSTELERDEN "ADET/SAYI" ÇIKARMA
def liste_sayisi_al(val):
    if pd.isna(val) or val == "" or val == "nan": return 0
    # Eğer veri bir liste görünümündeyse (örn: "['a', 'b']")
    if "[" in str(val):
        try:
            # Virgülleri sayarak eleman sayısını tahmin et (+1)
            return str(val).count(',') + 1
        except: return 1
    return 1 if val != 0 else 0

# Program ve Fonksiyon sayılarını yeni sütun yapalım
df['Program_Sayisi'] = df['Programlar'].apply(liste_sayisi_al)
df['Fonksiyon_Sayisi'] = df['Fonksiyonlar'].apply(liste_sayisi_al)

# 2. ADIM: ADET BELİRTEN STR SÜTUNLARI SAYIYA ÇEVİRME
adet_sutunlari = [
    'Dondurucu Çekmece Sayısı', 'Sebzelik Adedi', 'Fincan Raf Sayısı', 
    'Ön Kapak Cam Sayısı', 'Lamba Adedi', 'İşlemci Çekirdek Sayısı'
]

for col in adet_sutunlari:
    if col in df.columns:
        # Metin içindeki sayıları çek (Örn: "2 Adet" -> 2)
        df[col] = pd.to_numeric(df[col].astype(str).str.extract(r'(\d+)')[0], errors='coerce')

# 3. ADIM: EKRAN BOYUTU (INCH -> FLOAT)
if 'Ekran Boyutu' in df.columns:
    # "6.67 in" -> 6.67
    df['Ekran_Boyutu_Inch'] = df['Ekran Boyutu'].astype(str).str.replace(',', '.').str.extract(r'(\d+\.?\d*)')[0].astype(float)
    df.drop(columns=['Ekran Boyutu'], inplace=True)

# 4. ADIM: ÇOK SEYREK SÜTUNLARI SİLME (GÜLTÜ AZALTMA)
# 1301 satırda 20'den az veri varsa o özellik model için genelleme yapamaz.
limit = 20
seyrek_cols = [col for col in df.columns if df[col].count() < limit]
# Önemli olabilecekleri koruyalım
korunacaklar = ['urun_adi', 'fiyat', 'Main_Category', 'Subcategory', 'Guc (W)', 'Agirlik (kg)']
seyrek_cols = [c for c in seyrek_cols if c not in korunacaklar]

df.drop(columns=seyrek_cols, inplace=True)

# 5. ADIM: FİNAL VERİ TİPİ DÜZENLEME
# Bellek tasarrufu için 1/0 olanları int8 yapabilirsin
binary_cols = df.columns[df.isin([0, 1]).all()]
for col in binary_cols:
    df[col] = df[col].astype(np.int8)

print(f"Final Optimizasyon Tamamlandı! Yeni Sütun Sayısı: {len(df.columns)}")

Final Optimizasyon Tamamlandı! Yeni Sütun Sayısı: 286


In [255]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 1301 entries, 0 to 1300
Data columns (total 286 columns):
 #    Column                                                         Non-Null Count  Dtype  
---   ------                                                         --------------  -----  
 0    urun_linki                                                     1301 non-null   str    
 1    urun_adi                                                       1301 non-null   str    
 2    fiyat                                                          1176 non-null   float64
 3    enerji_sinifi                                                  270 non-null    str    
 4    favori_sayisi                                                  1237 non-null   Int64  
 5    Product_Group                                                  1301 non-null   str    
 6    Main_Category                                                  1301 non-null   str    
 7    Subcategory                                                 

In [256]:
df.sample(10, random_state=42)

,urun_linki,urun_adi,fiyat,enerji_sinifi,favori_sayisi,Product_Group,Main_Category,Subcategory,Derinlik,Genişlik,Yükseklik,Sıcaklık Yükselme Süresi (Saat),Dondurucu Yeri,Elektronik Gösterge,Kontrol Sistemi,Tatil Modu,ProSmart™ Inverter Kompresör,Aydınlatma Tipi,Kapı Yönü Değiştirme,İklim Sınıfı,Ses Seviyesi,Ses Seviyesi Sınıfı,Soğutma Sistemi,Soğutucu Bölme Fanı,EverFresh+ Teknolojisi,Kahvaltılık Çekmecesi,Dondurucu Çekmece Sayısı,Buzluk Tipi,Dondurucu Bölme Aydınlatma Tipi,Sebzelik Adedi,Yıllık Enerji Tüketimi (kWh),Günlük Enerji Tüketimi (kwh/Gün),Günlük enerji tüketimi 16°C (kWh/24h) (EU_2021_EP),Ürün Tipi,Hızlı Soğutma,Hızlı Dondurma Bölmesi,Kapasite,IronFast,Çocuk Kilidi,Deterjan Sistemi,Buhar,Motor Tipi,Tambur Deseni,Elektronik Su Kontrol Sistemi,Gösterge Tipi,Kalan Zaman Göstergesi,Su Taşma Emniyet Sistemi,Dengesiz Yük Kontrolü,Su Kesik İkazı,Programlar,Dinamik Rezistans,Bağlantı,Fonksiyonlar,Ekran Tipi,Paslanmaz Çelik İç Gövde,Otomatik Kapı Açma Özelliği,Bardak Koruma Sistemi,Yarım Yük Yıkama,Yüksekliği Ayarlanabilir Fincan Rafları,Fincan Raf Sayısı,Çatal Kaşık Sepeti,Enerji Tüketimi,Su Tüketimi,Parlatıcı Göstergesi,Tuz Göstergesi,Su Güvenlik Sistemi,Kir Sensörü,Dokunmatik Tuş,Direkt Su Tahliyesi,Su Tankı Dolu Uyarısı,Kondenser Temizleme Uyarısı,Fıltre Temizleme Uyarısı,Ürün Serisi,Fırın Hacmi,Fırın İçi Aydınlatma,Elektrikli Izgara,Ön Kapak Cam Sayısı,Elektroturbo (Fan Destekli Pişirme),Saat Tipi,Fan ve Alt Isıtıcı,Alt Isıtıcı,Fan Destekli Isıtıcı,Küçük Izgara,Kolayca Sökülebilir ve Temizlenebilir Komple İç Cam,Buharlı Pişirme,Doğalgaz,Ocak Yüzeyi,Kahve Beki Adaptörü,Gaz Emniyet Sistemi,3D Pişirme,FitFry Teknolojisi,Ekran Kilidi,Ocak Tipi Ve Göz Sayısı,Ocak Tipi,Bek Şapkası Tipi,Kontrol Tipi ve Yeri,Sol Ön Bölme,Sağ Arka Bölme,Tencere Izgara Tipi,Çerçeve,Sağ Ön Bölme,Sol Arka Bölme,Zamanlayıcı,Sıcaklık Ayarı,Bulaşık Makinesinde Yıkanabilir Aksesuarlar,LCD Ekran,Aşırı Isınma Emniyeti,Davlumbaz Tipi,Maksimum Çekiş Gücü,Yağ Filtre Adedi,Montaj Yüksekliği (mm),Bulaşık Makinesinde Yıkanabilir Filtre,Kontrol Tipi,Lamba Adedi,Yoğun Emiş Gücü,Otomatik Kapanma Özelliği,Kademe Sayısı,Filtre Tipi,Filtre Doluluk Göstergesi,Her Kademe İçin Ses Güçleri (DIN/EN 60704-3 standardına göre),Hava Temizleme Özelliği,Önden Ayarlanablir Arka Ayaklar,Yüksek Bazaya Montaj,İşletim Sistemi,USB Bağlantısı,HDMI,Bluetooth,Arka Kamera,Ön Kamera,İşlemci,Ekran Çözünürlüğü,Mikrofon,GPS,Kulaklık Tipi,NFC,Aux Girişi,Uzaktan Kumanda,İşlemci Çekirdek Sayısı,2.Arka Kamera,Kamera Zoom,Ön Kamera Flaş,Wi-Fi,Marka,Yüz Haritalama,Çift Hatlı,Ses Kayıt,Ürün Ailesi,İşletim Sistemi Versionu,Görüntülü Konuşma,Uyumlu olduğu modeller (MPA),Hızlı Şarj desteği,Type¬C veya Micro USB kablolarıyla şarj edilebilir,Su Geçirmezlik,Soğutma Kapasitesi,Isıtma Kapasitesi (Btu/h),Yıllık Tüketim (kW) Soğuma,Yıllık Tüketim Isıtma (kW),Voltaj,Soğutucu Akışkan,Mevsimsel (SCOP) Enerji Verimi Isıtma (W/W),Mevsimsel (SEER) Enerji Verimi Soğutma (W/W),Akıllı Çalışma Sistemi,Gizli Gösterge,Uyku Mode,Dış Ünite Soğutma Çalışma Aralığı(°C),Dış Ünite Isıtma Çalışma Aralığı(°C),Özel Filtreler,Otomatik Temizleme,Otomatik Hava Yönlendirme (Yukarı-Aşağı),4 Yönlü Otomatik Hava Kontrolü,Nem Alma,Enerji Sınıfı-Isıtma,Kapasite (Isıtma),Anlık Tüketim Göstergesi,Enerji Kontrol,Sessiz Mod,Yumuşak Hava,Auto Swing,Dış Ünite (GxYxD) mm,İç Ünite (GxYxD) mm,Su Tankı Kapasitesi,Devrilme Emniyeti,Titanyum Emaye Kaplamalı Kazan,Yüksek Basınç Emniyet Valfi,Susuz Çalışma Emniyeti,Donma Emniyeti,Korozyona Karşı Koruma,Yerden Isıtmaya Uygunluk,Oda Termostatı İle Çalışma,Alev Modülasyonu,Fan Modülasyonu,Arıza Teşhis Sistemi,Emniyet Termostatı,Emniyet Ventili,Hava Emiş Emniyeti,Alev Sönme Emniyeti,3 Yollu Vana Blokaj Sistemi,Otomatik By-Pass Sistemi,Pompa Anti Blokaj Sistemi,Fincan Kapasitesi,Işıklı Uyarı,Ölçü Kaşığı,Sesli Uyarı,Cook Sense Teknolojisi (Pişmeyi Algılama),Anti Spill Teknolojisi (Taşmayı Önleme),Tek Tuşla Kontrol,Fincan Boyutu Seçimi,Su Seviye Göstergesi,Otomatik Kapanma,Sıcak Tutma Özelliği,Kablo Sarma Y

# Şikayet Var

In [258]:
df = pd.read_csv("../web-scraping-data/data/raw/sikayet-var/sikayet_var_yorumlar.csv")
df.head()

,Yorum_Linki,Sikayet_Basligi,Sikayet,Sikayet_Tarihi,Goruntuleme_Sayisi,Destek_Sayisi,Kategori,Urun_Adi,Sirket_Cevabi,Gelisme_Tarih,Gelisme_Yorum,Cevap_Tarihi,Cevap_Mesaji,Cevaba_Cevap_1_Tarih,Cevaba_Cevap_1_Mesaj,Cevaba_Cevap_2_Tarih,Cevaba_Cevap_2_Mesaj,Cevaba_Cevap_3_Tarih,Cevaba_Cevap_3_Mesaj,Cevaba_Cevap_4_Tarih,Cevaba_Cevap_4_Mesaj,Cevaba_Cevap_5_Tarih,Cevaba_Cevap_5_Mesaj
0,https://www.sikayetvar.com/beko/ic-haznedeki-p...,İç Haznedeki Paslanma Sorunu TekrarlayanBekoFı...,2025 yılı Haziran ayında Esenyurt Beko bayisin...,28 Nisan 14:56,160,0,Ankastre Fırın,BFC 631,1,NaN,NaN,28 Nisan 15:49,"Sevgili müşterimiz ,\n\nÖncelikle Firmamıza gö...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.sikayetvar.com/beko/beko-magazasi-...,"BekoMağazası Büyük Ocağı Geri Almayı Reddetti,...",Kocaeli Gebze’de bulunan Beko Piranlar şubesin...,14 Nisan 21:12,225,0,Ocak,BFC 631,1,NaN,NaN,15 Nisan 11:55,"Sevgili müşterimiz ,\n\nÖncelikle Firmamıza gö...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.sikayetvar.com/beko/beko-fitfry-fi...,BekoFitfry Fırının Yanık Lekeleri Temizlenmiyor,Beko Fitfry bfc 631 s fırını 8 Kasım 2025 tari...,01 Nisan 14:58,379,0,Fırın,BFC 631,1,NaN,NaN,01 Nisan 17:54,"Sevgili müşterimiz ,\n\nÖncelikle Firmamıza gö...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.sikayetvar.com/beko/teslim-edilmey...,Teslim Edilmeyen Ürünler Ve Cevapsız Telefon A...,Beko markasının Sultanbeyli ilçesinde bulunan ...,02 Mart 00:48,854,0,Ankastre Fırın,BFC 631,1,NaN,NaN,02 Mart 14:40,"Sevgili müşterimiz ,\n\nÖncelikle Firmamıza gö...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.sikayetvar.com/beko/beko-ankastre-...,BekoAnkastre Fırında Özellik Eksikliği Ve Pişi...,Beko marka ankastre set kapsamında aldığım fır...,04 Şubat 00:04,1.114,1,Ankastre Fırın,BFC 631,1,NaN,NaN,04 Şubat 14:17,"Sevgili müşterimiz ,\n\nÖncelikle Firmamıza gö...",05 Şubat 20:34,05/02 Tarihinde fırın için teknisyeniniz geldi...,26 Şubat 17:01,Tüketici olarak Beko marka ürünmü almak asla.....,NaN,NaN,NaN,NaN,NaN,NaN


In [259]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 12367 entries, 0 to 12366
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Yorum_Linki           12367 non-null  str  
 1   Sikayet_Basligi       12367 non-null  str  
 2   Sikayet               12367 non-null  str  
 3   Sikayet_Tarihi        12367 non-null  str  
 4   Goruntuleme_Sayisi    12367 non-null  str  
 5   Destek_Sayisi         12367 non-null  int64
 6   Kategori              12367 non-null  str  
 7   Urun_Adi              12367 non-null  str  
 8   Sirket_Cevabi         12367 non-null  int64
 9   Gelisme_Tarih         1476 non-null   str  
 10  Gelisme_Yorum         1476 non-null   str  
 11  Cevap_Tarihi          11751 non-null  str  
 12  Cevap_Mesaji          11751 non-null  str  
 13  Cevaba_Cevap_1_Tarih  5683 non-null   str  
 14  Cevaba_Cevap_1_Mesaj  5683 non-null   str  
 15  Cevaba_Cevap_2_Tarih  1842 non-null   str  
 16  Cevaba_Cevap_2_

In [ ]:

def sayi_temizle(val):
    if pd.isna(val): 
        return 0
    
    # Metni standartlaştır: küçük harf, binlik ayracı (nokta) sil ve boşlukları at
    s = str(val).lower().replace('.', '').strip()
    
    # 1. Hatanın Kaynağı: '-' veya boş metin kontrolü
    if s == '-' or s == '' or not any(char.isdigit() for char in s):
        return 0
        
    # 2. Binlik Kısaltma Kontrolü (Örn: 1.2b veya 5k)
    # İşlem: $1.2b \rightarrow 1.2 \times 1000 = 1200$
    if 'b' in s or 'k' in s:
        try:
            temiz_s = s.replace('b', '').replace('k', '').replace(',', '.')
            return int(float(temiz_s) * 1000)
        except:
            return 0
            
    # 3. Standart Sayıya Dönüştürme
    try:
        # Virgül varsa noktaya çevirip float üzerinden int'e geçmek en garantisidir
        return int(float(s.replace(',', '.')))
    except ValueError:
        return 0

# --- VERİ TİPİ DÖNÜŞÜM MERKEZİ ---

# 1. Görüntüleme Sayısı (Hata Onarılmış)
if 'Goruntuleme_Sayisi' in df.columns:
    df['Goruntuleme_Sayisi'] = df['Goruntuleme_Sayisi'].apply(sayi_temizle)

# 2. Tarih Sütunlarını Optimize Et
tarih_sutunlari = [
    'Sikayet_Tarihi', 'Gelisme_Tarih', 'Cevap_Tarihi', 
    'Cevaba_Cevap_1_Tarih', 'Cevaba_Cevap_2_Tarih', 
    'Cevaba_Cevap_3_Tarih', 'Cevaba_Cevap_4_Tarih', 'Cevaba_Cevap_5_Tarih'
]

for col in tarih_sutunlari:
    if col in df.columns:
        # Hatalı tarihleri pas geçmek için errors='coerce' kullanıyoruz
        df[col] = pd.to_datetime(df[col], errors='coerce')

# 3. Kategori ve Ürün Adı (Bellek Tasarrufu)
# 12.367 satırda bu sütunları 'category' yapmak RAM kullanımını %50 düşürür
for col in ['Kategori', 'Urun_Adi']:
    if col in df.columns:
        df[col] = df[col].astype('category')

# 4. Şirket Cevabı (1/0 ise En Küçük Tamsayı Tipi)
if 'Sirket_Cevabi' in df.columns:
    df['Sirket_Cevabi'] = df['Sirket_Cevabi'].astype('int8')

print("Veri tipleri başarıyla güncellendi! Hatalı karakterler temizlendi.")

Veri tipleri başarıyla güncellendi! Hatalı karakterler temizlendi.


In [262]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12367 entries, 0 to 12366
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype        
---  ------                --------------  -----        
 0   Yorum_Linki           12367 non-null  str          
 1   Sikayet_Basligi       12367 non-null  str          
 2   Sikayet               12367 non-null  str          
 3   Sikayet_Tarihi        0 non-null      datetime64[s]
 4   Goruntuleme_Sayisi    12367 non-null  int64        
 5   Destek_Sayisi         12367 non-null  int64        
 6   Kategori              12367 non-null  category     
 7   Urun_Adi              12367 non-null  category     
 8   Sirket_Cevabi         12367 non-null  int8         
 9   Gelisme_Tarih         0 non-null      datetime64[s]
 10  Gelisme_Yorum         1476 non-null   str          
 11  Cevap_Tarihi          0 non-null      datetime64[s]
 12  Cevap_Mesaji          11751 non-null  str          
 13  Cevaba_Cevap_1_Tarih  0 non-null      date